# Few-Shot CryoET Particle Detection with Weak Labels and Volume Infill

## Part 0 — Overview and Research Background

### 0.1 Background

Cryo-electron tomography (CryoET) provides three-dimensional views of macromolecular structures in their near-native cellular environments. To analyze these structures, researchers often need to locate many copies of the same particle inside a reconstructed tomogram. This process is known as **3D particle picking** or **particle detection**.

Automatic CryoET particle detection remains challenging because tomograms commonly contain:

* very low signal-to-noise ratios;
* missing-wedge distortions;
* crowded and non-uniform cellular backgrounds;
* sparsely distributed target particles;
* very limited manually annotated data.

Although deep-learning models can improve particle localization, conventional supervised methods often require many labeled particles. Creating these annotations is difficult because particles may be faint, distorted, and hard to distinguish from surrounding structures.

This tutorial therefore considers a **few-shot detection setting**: only a small number of particle centers are annotated, and these annotations are used to train a lightweight model that locates additional particles in a three-dimensional tomogram.

---

### 0.2 Tutorial Objective

The central question of this tutorial is:

> **How can a small number of particle-center annotations be converted into useful training data for 3D CryoET particle detection?**

To answer this question, the tutorial will demonstrate the following pipeline:

**Synthetic CryoET-like tomogram → few-shot center annotations → spherical weak labels → training subvolumes → Volume Infill → lightweight 3D detector → probability volume → particle coordinates → detection evaluation**

The tutorial focuses on three main ideas:

1. **Weak labels**

   Instead of manually drawing an accurate 3D boundary for every particle, each annotated particle center is converted into a small spherical label. This provides an approximate voxel-level training target while keeping the annotation process simple.

2. **Few-shot training**

   Only a small number of annotated particles are used to construct the original training set. This setting allows us to observe the problems caused by limited labels and sparse particle distributions.

3. **Volume Infill**

   Particle patches extracted from the few annotated examples are transformed and inserted into new background locations. This increases the number, spatial distribution, and appearance variation of particles available during training.

The resulting model will predict a voxel-wise probability volume. Connected-component analysis will then convert the predicted particle regions into center coordinates. The final detections will be evaluated using precision, recall, F1-score, and localization error.

---

### 0.3 Scope of This Tutorial

This notebook is a **small educational implementation**, not a complete reproduction of any of the referenced methods.

To keep the workflow understandable and computationally manageable, the tutorial will use:

* a synthetic CryoET-like tomogram;
* one particle class;
* a small number of center annotations;
* spherical weak labels;
* particle-centered training subvolumes;
* spatial particle augmentation and Volume Infill;
* a lightweight 3D convolutional detector;
* connected components for particle-center extraction;
* coordinate-based detection metrics.

The notebook will not implement multi-class particle detection, large-scale real-tomogram training, SimCLR pretraining, self-interpreted consistency guidance, or the complete architectures and post-processing systems proposed in the original papers.

---

### 0.4 Reference Papers

This tutorial is primarily based on the following studies.

#### 1. SaSi — Main Methodological Reference

**G. Adethya, B. P. Mantha, T. Wang, X. Li, and M. Xu,
“SaSi: A Self-augmented and Self-interpreted Deep Learning Approach for Few-shot Cryo-ET Particle Detection,” 2025.**

[Paper: arXiv:2505.19948](https://arxiv.org/abs/2505.19948)

SaSi is the main methodological reference for this tutorial. It studies CryoET particle detection under few-shot conditions and addresses two important problems: the scarcity of labeled particles and the sparse distribution of particles in tomograms.

This tutorial adapts the following ideas from SaSi:

* few-shot particle-center annotations;
* spherical labels generated around annotated centers;
* particle-centered training samples;
* spatial transformation of annotated particles;
* self-augmented Volume Infill;
* voxel-wise particle prediction;
* connected-component-based particle localization;
* coordinate-based detection evaluation.

SaSi also introduces self-supervised pretraining and self-interpreted consistency guidance. These components are outside the implementation scope of this notebook and will only be mentioned as possible extensions.

---

#### 2. DeepETPicker — Weak-Label Detection Workflow

**G. Liu, T. Niu, M. Qiu, et al.,
“DeepETPicker: Fast and Accurate 3D Particle Picking for Cryo-Electron Tomography Using Weakly Supervised Deep Learning,” Nature Communications, vol. 15, article 2090, 2024.**

[Paper: Nature Communications](https://www.nature.com/articles/s41467-024-46041-0)

DeepETPicker demonstrates how simplified particle labels can support efficient 3D particle picking without requiring accurate full-particle masks.

This tutorial uses DeepETPicker as a reference for the general detection workflow:

* converting manually annotated centers into simplified weak labels;
* dividing a tomogram into smaller training subvolumes;
* using a 3D segmentation-style network to produce voxel predictions;
* performing overlapping whole-volume inference;
* converting predicted particle regions into center coordinates.

The complete 3D-ResUNet architecture and MP-NMS post-processing method proposed by DeepETPicker will not be reproduced.

---

#### 3. DeepFinder — Background Reference

**E. Moebel et al.,
“Deep Learning Improves Macromolecule Identification in 3D Cellular Cryo-Electron Tomograms,” Nature Methods, vol. 18, pp. 1386–1394, 2021.**

[Paper: Nature Methods](https://www.nature.com/articles/s41592-021-01275-4)

DeepFinder is included as a background reference because it established an important segmentation-based approach to macromolecule localization in CryoET data.

Its general workflow can be summarized as:

**3D tomogram → voxel-wise segmentation → spatial grouping → particle coordinates**

This idea helps explain why the model in this tutorial first predicts a three-dimensional probability map instead of directly predicting particle coordinates. DeepFinder itself will not be implemented or reproduced.

---

### 0.5 Relationship Between the References and This Tutorial

| Reference        | Role in this tutorial                | Implemented components                                                                                     |
| ---------------- | ------------------------------------ | ---------------------------------------------------------------------------------------------------------- |
| **SaSi**         | Main methodological reference        | Few-shot setting, spherical labels, particle augmentation, Volume Infill, connected-component localization |
| **DeepETPicker** | Weakly supervised detection workflow | Training subvolumes, voxel-wise prediction, whole-volume inference, coordinate extraction                  |
| **DeepFinder**   | Background and historical context    | Segmentation-to-localization concept only                                                                  |

The final notebook should therefore be understood as a:

> **SaSi-inspired educational particle-detection pipeline, supported by the weak-label workflow of DeepETPicker and the segmentation-to-localization concept established by DeepFinder.**

---

### 0.6 Tutorial Structure

The remaining parts of the notebook are organized as follows:

* **Part 1:** CryoET Particle Data and Few-Shot Weak Labels
* **Part 2:** Building Few-Shot Training Subvolumes
* **Part 3:** Self-Augmented Volume Infill
* **Part 4:** Training a Lightweight 3D Detector
* **Part 5:** Whole-Volume Inference and Particle Localization
* **Part 6:** Detection Evaluation and Volume Infill Comparison
* **Part 7:** Summary, Limitations, and Outlook

Part 0 introduces the research basis and tutorial scope. The practical workflow begins in Part 1.


## Part 1 — CryoET Particle Data and Few-Shot Weak Labels

Before training a particle detector, we first need to understand what information is contained in a three-dimensional tomogram and how a small number of manual annotations can be converted into training labels.

In this part, we will create a small synthetic CryoET-like tomogram, inspect its particles from different viewing directions, select a few particle centers as annotations, and convert those point annotations into spherical weak labels.

---

### 1.1 From a 3D Tomogram to Particle Coordinates

A CryoET tomogram can be represented as a three-dimensional array:

$$
V \in \mathbb{R}^{D \times H \times W}
$$

where:

* $D$ is the number of slices along the depth direction;
* $H$ is the height of each slice;
* $W$ is the width of each slice;
* each element of $V$ is a **voxel** containing an image-intensity value.

In this notebook, a voxel coordinate is written in the order:

$$
\mathbf{c}=(z,y,x)
$$

This ordering follows the indexing convention used by NumPy and PyTorch for a three-dimensional volume:

```text
volume[z, y, x]
```

A tomogram can be inspected through three orthogonal viewing planes:

| Plane | Fixed coordinate | Displayed axes                 |
| ----- | ---------------- | ------------------------------ |
| XY    | $z$              | horizontal: $x$, vertical: $y$ |
| XZ    | $y$              | horizontal: $x$, vertical: $z$ |
| YZ    | $x$              | horizontal: $y$, vertical: $z$ |

A single particle usually appears across several neighboring slices. Its appearance may change between slices because different planes intersect different parts of the three-dimensional structure. For this reason, particle detection should use volumetric information instead of treating every slice as an independent image.

A maximum-intensity projection can also summarize a volume along one direction. For example, an XY projection is calculated by:

$$
P_{XY}(y,x)=\max_z V(z,y,x)
$$

Although projections are useful for visualization, they collapse depth information. The detector must therefore operate on the original 3D volume.

---

### 1.2 Particle Detection and Segmentation Are Different Tasks

In the previous interactive segmentation tutorial, the objective was to identify the complete three-dimensional region occupied by a selected target. The output was therefore a 3D segmentation mask.

In particle detection, the main objective is different. Instead of recovering the exact boundary of every particle, we want to estimate the center of each target particle:

$$
\mathcal{C}={\mathbf{c}_1,\mathbf{c}_2,\ldots,\mathbf{c}_K}
$$

where $K$ is the number of particles and each $\mathbf{c}_i=(z_i,y_i,x_i)$ is a particle-center coordinate.

| Task                    | Expected output                                  |
| ----------------------- | ------------------------------------------------ |
| 3D segmentation         | A voxel-wise mask describing the target boundary |
| 3D particle detection   | A set of particle-center coordinates             |
| Particle classification | A class label assigned to each detected particle |

The detector developed in this notebook will temporarily predict a voxel-wise probability volume. However, this probability volume is only an intermediate representation. It will later be converted into particle-center coordinates.

---

### 1.3 Few-Shot Center Annotations

Accurately drawing the full boundary of a particle in a low-SNR tomogram is difficult and time-consuming. Marking an approximate particle center is much simpler.

In an $N$-shot setting, only $N$ particle centers are provided as training annotations:

$$
\mathcal{C}_{\mathrm{shot}}
===========================

{\mathbf{c}_1,\mathbf{c}_2,\ldots,\mathbf{c}_N}
\subset
\mathcal{C}
$$

For example, in a 5-shot experiment, the tomogram may contain many particles, but only five centers are available to the training process.

It is important to distinguish between two coordinate sets:

* **Ground-truth centers:** all particle centers in the synthetic tomogram, retained only for visualization and final evaluation;
* **Few-shot annotations:** the small subset of centers that the model is allowed to use during training.

The unselected particles must not be added to the training labels. Otherwise, the experiment would no longer represent a few-shot setting.

Another important consequence is that a voxel outside the few-shot labels is not automatically guaranteed to be true background. It may belong to an unannotated particle. We will return to this issue when constructing training subvolumes in Part 2.

---

### 1.4 Converting Points into Spherical Weak Labels

A center coordinate contains no spatial extent, while a voxel-wise detector requires a target value for each voxel. We therefore expand every annotated center into a small spherical region.

For an annotated center $\mathbf{c}_i$, the weak label is defined as:

$$
Y_i(\mathbf{v})=
\begin{cases}
1, & |\mathbf{v}-\mathbf{c}_i|*2 \leq r*{\mathrm{label}},\
0, & \text{otherwise},
\end{cases}
$$

where:

* $\mathbf{v}=(z,y,x)$ is a voxel coordinate;
* $\mathbf{c}_i$ is an annotated particle center;
* $r_{\mathrm{label}}$ is the weak-label radius;
* $|\cdot|_2$ is the Euclidean distance.

For multiple annotated particles, the complete weak-label volume is the union of all spherical labels:

$$
Y(\mathbf{v})=\max_{i=1,\ldots,N}Y_i(\mathbf{v})
$$

This process converts a small set of point annotations into a voxel-wise training target.

However, the resulting sphere is a **weak label**, not an accurate particle segmentation. It only indicates a small region around the expected particle center.

---

### 1.5 Choosing the Weak-Label Radius

The value of $r_{\mathrm{label}}$ determines how much of the area around each annotated center is treated as the positive class.

* If the radius is too small, very few positive voxels are available and the model may struggle to learn a stable particle response.
* If the radius is too large, the label may include substantial background or overlap with neighboring particles.
* A moderate radius provides a visible training region while keeping the label concentrated around the particle center.

In this tutorial, the default weak-label radius is smaller than the approximate synthetic particle radius. This encourages the detector to identify the central region of a particle rather than reproduce its exact boundary.

Because the final objective is particle localization, a useful weak label does not need to match the true particle shape perfectly.

---

## Task 1 — Explore Particles and Create Few-Shot Weak Labels

### Scenario

Suppose a small CryoET-like tomogram contains several copies of the same particle. The complete particle coordinates are known because the volume is synthetically generated, but only a few of these coordinates will be treated as manual annotations.

Your goal is to inspect the volume, understand its coordinate system, select a small annotation subset, and convert those point annotations into spherical weak labels.

### What You Will Do

In the following code cell, you will:

1. generate a reproducible synthetic CryoET-like tomogram;
2. inspect XY, XZ, and YZ slices;
3. display a projection of the particle volume;
4. examine the complete ground-truth particle coordinates;
5. select $N$ centers as few-shot annotations;
6. generate spherical weak labels around the selected centers;
7. compare particle appearance, point annotations, and weak-label regions;
8. verify that only the selected few-shot particles appear in the weak-label volume.

### Main Editable Parameters

| Parameter         |  Default value | Meaning                                            |
| ----------------- | -------------: | -------------------------------------------------- |
| `SEED`            |           `42` | Controls reproducible particle placement and noise |
| `VOLUME_SHAPE`    | `(64, 64, 64)` | Depth, height, and width of the synthetic volume   |
| `NUM_PARTICLES`   |           `18` | Total number of particles in the tomogram          |
| `PARTICLE_RADIUS` |            `4` | Approximate radius of a synthetic particle         |
| `NOISE_STD`       |         `0.35` | Standard deviation of the added background noise   |
| `N_SHOT`          |            `5` | Number of particle centers used as annotations     |
| `LABEL_RADIUS`    |            `3` | Radius of each spherical weak label                |

Change one parameter at a time when exploring the results. For example:

* increase `NOISE_STD` to make the particles more difficult to observe;
* change `N_SHOT` to compare 3-shot, 5-shot, and 10-shot annotation settings;
* adjust `LABEL_RADIUS` to observe how the weak-label volume changes.

### Expected Results

After completing the task, you should obtain:

* a synthetic volume with multiple particles embedded in noisy background;
* orthogonal slice views showing the same 3D structures from different directions;
* a list of all ground-truth particle centers;
* a smaller list containing only the selected few-shot annotations;
* a 3D weak-label volume containing one sphere per annotated particle;
* visual overlays confirming that every weak-label sphere is centered on its corresponding annotation.

At the end of this task, you should be able to explain why a particle-center coordinate, a spherical weak label, and a true particle boundary represent three different types of information.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Task 1 — Construct a synthetic few-shot CryoET problem
# ============================================================

# Learning goal:
# Create a noisy 3D tomogram containing multiple particles, retain only a
# small number of center annotations, and convert them into spherical weak labels.

# WriteYourCodeHere: change one parameter at a time and inspect the result.
SEED = 42
VOLUME_SHAPE = (64, 64, 64)
NUM_PARTICLES = 18
N_SHOT = 5
PARTICLE_RADIUS = 4
WEAK_LABEL_RADIUS = 2
NOISE_STRENGTH = 0.45
DISPLAY_INTERPOLATION = "bilinear"

# Internal settings used to generate stable synthetic data.
CENTER_MARGIN = 9
MIN_FEW_SHOT_SEPARATION = 16
MIN_PARTICLE_DISTANCE = 8
POSITIVE_CROP_HALF = 8
SIGNAL_SUPPORT_RADIUS = 6
MAX_CENTER_ATTEMPTS = 50000

rng = np.random.default_rng(SEED)


# ============================================================
# 1. Generate particle-center coordinates
# ============================================================

def propose_center():
    """Generate a valid integer center in (z, y, x) order."""
    return np.array([
        rng.integers(CENTER_MARGIN, axis_size - CENTER_MARGIN)
        for axis_size in VOLUME_SHAPE
    ], dtype=int)


# Generate the five annotated particles first and keep them well separated.
few_shot_center_list = []

for _ in range(MAX_CENTER_ATTEMPTS):
    candidate = propose_center()

    sufficiently_separated = all(
        np.max(np.abs(candidate - previous)) >= MIN_FEW_SHOT_SEPARATION
        for previous in few_shot_center_list
    )

    if sufficiently_separated:
        few_shot_center_list.append(candidate)

    if len(few_shot_center_list) == N_SHOT:
        break

if len(few_shot_center_list) < N_SHOT:
    raise RuntimeError("Could not generate enough separated few-shot centers.")


# Generate the remaining unannotated particles.
# They are prevented from entering the future 16³ positive training crops.
all_center_list = few_shot_center_list.copy()

for _ in range(MAX_CENTER_ATTEMPTS):
    candidate = propose_center()

    outside_all_positive_crops = all(
        not np.all(
            np.abs(candidate - annotated_center)
            <= POSITIVE_CROP_HALF + SIGNAL_SUPPORT_RADIUS
        )
        for annotated_center in few_shot_center_list
    )

    separated_from_existing_particles = all(
        np.linalg.norm(candidate - existing_center) >= MIN_PARTICLE_DISTANCE
        for existing_center in all_center_list
    )

    if outside_all_positive_crops and separated_from_existing_particles:
        all_center_list.append(candidate)

    if len(all_center_list) == NUM_PARTICLES:
        break

if len(all_center_list) < NUM_PARTICLES:
    raise RuntimeError("Could not generate the requested number of particle centers.")

ground_truth_centers = np.asarray(all_center_list, dtype=int)
few_shot_centers = np.asarray(few_shot_center_list, dtype=int)
few_shot_indices = np.arange(N_SHOT)
unannotated_centers = ground_truth_centers[N_SHOT:]


# ============================================================
# 2. Generate anisotropic synthetic particle signals
# ============================================================

def make_particle_template(random_generator, radius):
    """Create one asymmetric 3D particle with a random orientation."""
    coordinates = np.arange(-radius, radius + 1, dtype=np.float32)
    z, y, x = np.meshgrid(coordinates, coordinates, coordinates, indexing="ij")
    points = np.stack([z, y, x], axis=-1)

    rotation, _ = np.linalg.qr(random_generator.normal(size=(3, 3)))

    if np.linalg.det(rotation) < 0:
        rotation[:, 0] *= -1

    rotated_points = points @ rotation

    sigmas = random_generator.uniform(
        low=[1.7, 2.2, 1.7],
        high=[2.4, 3.2, 2.4]
    )

    primary_lobe = np.exp(
        -0.5 * np.sum((rotated_points / sigmas) ** 2, axis=-1)
    )

    shifted_points = rotated_points.copy()
    shifted_points[..., 1] -= random_generator.uniform(1.2, 2.2)

    secondary_lobe = 0.45 * np.exp(
        -0.5 * np.sum(
            (
                shifted_points
                / (sigmas * np.array([0.9, 0.8, 0.9]))
            ) ** 2,
            axis=-1
        )
    )

    template = primary_lobe + secondary_lobe
    radial_distance = np.sqrt(z ** 2 + y ** 2 + x ** 2)
    template[radial_distance > radius] = 0
    template /= template.max()

    return template.astype(np.float32)


clean_signal = np.zeros(VOLUME_SHAPE, dtype=np.float32)
support = SIGNAL_SUPPORT_RADIUS

for center in ground_truth_centers:
    particle = make_particle_template(rng, support)
    particle *= rng.uniform(1.8, 2.4)

    z, y, x = center
    clean_signal[
        z - support:z + support + 1,
        y - support:y + support + 1,
        x - support:x + support + 1
    ] += particle


# ============================================================
# 3. Add CryoET-like correlated and fine noise
# ============================================================

white_noise = rng.normal(size=VOLUME_SHAPE).astype(np.float32)

frequency_z = np.fft.fftfreq(VOLUME_SHAPE[0])[:, None, None]
frequency_y = np.fft.fftfreq(VOLUME_SHAPE[1])[None, :, None]
frequency_x = np.fft.fftfreq(VOLUME_SHAPE[2])[None, None, :]

anisotropic_filter = np.exp(
    -(
        (frequency_z / 0.10) ** 2
        + (frequency_y / 0.16) ** 2
        + (frequency_x / 0.16) ** 2
    )
)

correlated_noise = np.fft.ifftn(
    np.fft.fftn(white_noise) * anisotropic_filter
).real.astype(np.float32)

correlated_noise = (
    correlated_noise - correlated_noise.mean()
) / (correlated_noise.std() + 1e-8)

fine_noise = rng.normal(size=VOLUME_SHAPE).astype(np.float32)

tomogram = (
    clean_signal
    + NOISE_STRENGTH * correlated_noise
    + 0.12 * fine_noise
)

tomogram = (
    (tomogram - tomogram.mean())
    / (tomogram.std() + 1e-8)
).astype(np.float32)


# ============================================================
# 4. Convert few-shot centers into spherical weak labels
# ============================================================

weak_labels = np.zeros(VOLUME_SHAPE, dtype=np.uint8)
z_grid, y_grid, x_grid = np.ogrid[
    :VOLUME_SHAPE[0],
    :VOLUME_SHAPE[1],
    :VOLUME_SHAPE[2]
]

for z, y, x in few_shot_centers:
    spherical_region = (
        (z_grid - z) ** 2
        + (y_grid - y) ** 2
        + (x_grid - x) ** 2
        <= WEAK_LABEL_RADIUS ** 2
    )

    weak_labels[spherical_region] = 1


# ============================================================
# 5. Inspect the complete few-shot training scenario
# ============================================================

central_z = VOLUME_SHAPE[0] // 2
display_min, display_max = np.percentile(
    tomogram[central_z],
    [1, 99]
)

fig = plt.figure(figsize=(10, 4.5), dpi=140)

# Left: one observed tomogram slice.
ax_slice = fig.add_subplot(1, 2, 1)

image = ax_slice.imshow(
    tomogram[central_z],
    cmap="magma",
    origin="lower",
    interpolation=DISPLAY_INTERPOLATION,
    vmin=display_min,
    vmax=display_max
)

ax_slice.set_title(
    f"Observed tomogram — central XY slice\n"
    f"$z={central_z}$"
)
ax_slice.set_xlabel("x")
ax_slice.set_ylabel("y")

colorbar = fig.colorbar(image, ax=ax_slice, fraction=0.046, pad=0.04)
colorbar.set_label("Normalized intensity")


# Right: all synthetic particles and the five available annotations.
ax_centers = fig.add_subplot(1, 2, 2, projection="3d")

ax_centers.scatter(
    unannotated_centers[:, 2],
    unannotated_centers[:, 1],
    unannotated_centers[:, 0],
    s=45,
    facecolors="none",
    edgecolors="#64748b",
    linewidths=1.4,
    label="Unannotated particles"
)

ax_centers.scatter(
    few_shot_centers[:, 2],
    few_shot_centers[:, 1],
    few_shot_centers[:, 0],
    s=85,
    marker="*",
    color="#06b6d4",
    edgecolors="black",
    linewidths=0.5,
    label="Few-shot annotations"
)

ax_centers.set_xlim(0, VOLUME_SHAPE[2])
ax_centers.set_ylim(0, VOLUME_SHAPE[1])
ax_centers.set_zlim(0, VOLUME_SHAPE[0])
ax_centers.set_xlabel("x")
ax_centers.set_ylabel("y")
ax_centers.set_zlabel("z")
ax_centers.set_title(
    "Synthetic particle centers\n"
    "(complete centers are reference only)"
)
ax_centers.legend(loc="upper left", fontsize=8)
ax_centers.set_box_aspect(
    (VOLUME_SHAPE[2], VOLUME_SHAPE[1], VOLUME_SHAPE[0])
)

plt.tight_layout()
plt.show()


# ============================================================
# 6. Inspect one annotated particle and its weak label
# ============================================================

def extract_display_crop(volume, center, size):
    """Extract an odd-sized local crop centered at (z, y, x)."""
    half = size // 2
    start = np.asarray(center) - half
    end = start + size
    crop_slices = tuple(
        slice(start_value, end_value)
        for start_value, end_value in zip(start, end)
    )

    return volume[crop_slices].copy()


DISPLAY_CROP_SIZE = 21
display_half = DISPLAY_CROP_SIZE // 2
example_center = few_shot_centers[0]

example_input = extract_display_crop(
    tomogram,
    example_center,
    DISPLAY_CROP_SIZE
)

example_label = extract_display_crop(
    weak_labels,
    example_center,
    DISPLAY_CROP_SIZE
)

input_views = [
    example_input[display_half, :, :],
    example_input[:, display_half, :],
    example_input[:, :, display_half]
]

label_views = [
    example_label[display_half, :, :],
    example_label[:, display_half, :],
    example_label[:, :, display_half]
]

plane_names = ["XY", "XZ", "YZ"]
x_axis_labels = ["x", "x", "y"]
y_axis_labels = ["y", "z", "z"]

crop_min, crop_max = np.percentile(example_input, [2, 98])
fig, axes = plt.subplots(2, 3, figsize=(10, 6), dpi=140)

for column, plane_name in enumerate(plane_names):
    # Top row: the noisy input that will be provided to the detector.
    axes[0, column].imshow(
        input_views[column],
        cmap="magma",
        origin="lower",
        interpolation=DISPLAY_INTERPOLATION,
        vmin=crop_min,
        vmax=crop_max
    )

    axes[0, column].scatter(
        display_half,
        display_half,
        marker="+",
        s=100,
        color="cyan",
        linewidths=2
    )

    # Bottom row: the same input with the spherical weak-label boundary.
    axes[1, column].imshow(
        input_views[column],
        cmap="magma",
        origin="lower",
        interpolation=DISPLAY_INTERPOLATION,
        vmin=crop_min,
        vmax=crop_max
    )

    axes[1, column].contour(
        label_views[column],
        levels=[0.5],
        colors="cyan",
        linewidths=2
    )

    axes[1, column].scatter(
        display_half,
        display_half,
        marker="+",
        s=100,
        color="cyan",
        linewidths=2
    )

    axes[0, column].set_title(
        f"Observed model input — {plane_name}"
    )

    axes[1, column].set_title(
        f"Weak-label overlay — {plane_name}"
    )

    for row in range(2):
        axes[row, column].set_xlabel(x_axis_labels[column])
        axes[row, column].set_ylabel(y_axis_labels[column])
        axes[row, column].set_aspect("equal")

fig.suptitle(
    "One annotated particle: observed input and available supervision\n"
    "cyan + = annotated center; cyan boundary = spherical weak label",
    y=1.03
)

plt.tight_layout()
plt.show()


# ============================================================
# Checkpoint
# ============================================================

assert tomogram.shape == VOLUME_SHAPE
assert clean_signal.shape == VOLUME_SHAPE
assert weak_labels.shape == VOLUME_SHAPE
assert ground_truth_centers.shape == (NUM_PARTICLES, 3)
assert few_shot_centers.shape == (N_SHOT, 3)
assert set(np.unique(weak_labels)).issubset({0, 1})

# Every annotated center must lie inside its spherical weak label.
assert all(
    weak_labels[z, y, x] == 1
    for z, y, x in few_shot_centers
)

# No unannotated particle is allowed to enter a future 16³ positive crop.
for annotated_center in few_shot_centers:
    other_centers = ground_truth_centers[
        np.any(
            ground_truth_centers != annotated_center,
            axis=1
        )
    ]

    assert not np.any(
        np.all(
            np.abs(other_centers - annotated_center)
            <= POSITIVE_CROP_HALF + SIGNAL_SUPPORT_RADIUS,
            axis=1
        )
    )

weak_label_voxels = int(weak_labels.sum())
weak_label_fraction = weak_label_voxels / weak_labels.size

print(f"Tomogram shape: {tomogram.shape}")
print(f"Total synthetic particles: {NUM_PARTICLES}")
print(f"Available few-shot annotations: {N_SHOT}")
print(f"Unannotated particles: {NUM_PARTICLES - N_SHOT}")
print(f"Weak-label voxels: {weak_label_voxels:,}")
print(f"Weak-label fraction: {100 * weak_label_fraction:.4f}%")
print("Checkpoint passed: the few-shot centers and spherical weak labels are valid.")

## Part 2 — Building Few-Shot Training Subvolumes

In Part 1, we created a synthetic tomogram, selected a small number of particle-center annotations, and converted them into spherical weak labels. However, the complete 3D volume is not yet organized into a form suitable for model training.

In this part, we will extract smaller cubic regions called **training subvolumes**. Each input subvolume must be paired with a weak-label subvolume taken from exactly the same spatial location.

---

### 2.1 Why Use Training Subvolumes?

Real CryoET tomograms are usually too large to process as a single model input. A common solution is to divide the tomogram into smaller cubic regions.

For a cubic subvolume size $S$, an input sample has the form:

$$
X_i \in \mathbb{R}^{S \times S \times S}
$$

Its corresponding weak-label sample has the same spatial dimensions:

$$
Y_i \in {0,1}^{S \times S \times S}
$$

The model will later learn a mapping:

$$
f_\theta:X_i\rightarrow \hat{Y}_i
$$

where $\hat{Y}_i$ is a voxel-wise particle probability map.

The input and label must be extracted using identical coordinates. If the image crop and label crop are shifted relative to one another, the model receives an incorrect training target.

[DeepETPicker](https://www.nature.com/articles/s41467-024-46041-0) follows this general strategy by dividing tomograms into smaller cubic volumes and extracting particle-centered subvolumes for training.

---

### 2.2 Positive, Background, and Unlabeled Regions

The few-shot setting creates three different types of spatial information:

| Region type             | What is known?                                      | How is it used?                                 |
| ----------------------- | --------------------------------------------------- | ----------------------------------------------- |
| **Annotated positive**  | An annotated particle center is present             | Used to create a positive training subvolume    |
| **Verified background** | No target particle is present in the sampled region | Used to create a background training subvolume  |
| **Unlabeled region**    | No annotation is available                          | Must not automatically be treated as background |

The final distinction is important:

> **Unannotated does not mean negative.**

The synthetic tomogram contains particles that were not selected as few-shot annotations. These particles are unknown from the perspective of the training labels, but they are still real particles.

If a background subvolume is sampled without checking its contents, it may contain one of these unannotated particles. The model would then receive contradictory information: a particle-like structure in the input but an all-zero label.

In this educational synthetic experiment, the complete ground-truth coordinates will be used only to verify that a proposed background crop is empty. The unannotated centers will not be converted into positive training labels.

For real CryoET data, safe background regions may instead be manually verified or selected using additional expert knowledge.

---

### 2.3 Particle-Centered Positive Samples

For every few-shot center $\mathbf{c}_i=(z_i,y_i,x_i)$, we extract a cubic subvolume around that coordinate.

For an even subvolume size $S$, define:

$$
h=\frac{S}{2}
$$

The input crop is approximately:

$$
X_i=
V[
z_i-h:z_i+h,,
y_i-h:y_i+h,,
x_i-h:x_i+h
]
$$

The corresponding label crop is:

$$
Y_i=
Y[
z_i-h:z_i+h,,
y_i-h:y_i+h,,
x_i-h:x_i+h
]
$$

Both crops must have shape:

$$
S\times S\times S
$$

With this indexing convention, the annotated center appears at the local coordinate:

$$
(h,h,h)
$$

Particle-centered sampling guarantees that every available few-shot annotation contributes one positive training example. It also prevents the few positive particles from being missed by completely random sampling.

The subvolume size should be:

* large enough to contain the particle and some surrounding context;
* small enough to limit memory and computation;
* compatible with the downsampling stages of the later 3D network.

This tutorial will use a default size of:

$$
S=24
$$

which is large enough for the synthetic particles while remaining practical for CPU training.

---

### 2.4 Verified Background Samples

A detector must learn not only what a particle looks like, but also what background structures look like. We therefore add a small number of verified background subvolumes.

A valid background crop must satisfy two conditions:

1. it remains completely inside the tomogram boundaries;
2. it does not intersect any known synthetic particle region.

The corresponding background label is an all-zero volume:

$$
Y_{\mathrm{bg}}(\mathbf{v})=0
$$

To keep the sample-level dataset balanced, we will initially extract the same number of background subvolumes as positive subvolumes:

$$
N_{\mathrm{background}}=N_{\mathrm{shot}}
$$

This does not eliminate voxel-level class imbalance. It only prevents the training set from containing an excessive number of completely empty samples.

---

### 2.5 Voxel-Level Class Imbalance

Even a particle-centered subvolume contains many more background voxels than positive voxels.

For a training-label collection ${Y_i}_{i=1}^{N}$, the positive voxel fraction is:

$$
\rho=
\frac{
\sum_{i=1}^{N}\sum_{\mathbf{v}}Y_i(\mathbf{v})
}{
N S^3
}
$$

Because each weak label occupies only a small sphere, $\rho$ is usually much smaller than 1.

For example, increasing the number of background samples further would reduce $\rho$ and encourage a model to predict background everywhere. This is why later training will use losses designed for foreground imbalance instead of relying only on ordinary voxel accuracy.

At this stage, we will measure the imbalance rather than attempt to solve it. Dice-based and focal loss terms will be introduced in Part 4.

---

### 2.6 Preserving Data Separation

Overlapping subvolumes extracted from the same tomogram are strongly related. Randomly assigning these overlapping crops to training and testing sets could allow nearly identical voxel patterns to appear in both sets.

To avoid this leakage, the notebook will use volume-level separation:

| Dataset        | Source                                                 | Purpose                                    |
| -------------- | ------------------------------------------------------ | ------------------------------------------ |
| **Training**   | Current synthetic tomogram                             | Build few-shot and Volume Infill samples   |
| **Validation** | Independently generated tomogram with a different seed | Inspect model learning and select settings |
| **Test**       | Another independently generated tomogram               | Final whole-volume detection evaluation    |

Task 2 will therefore construct only the training subvolumes. Validation and test volumes will not be used to produce training samples.

---

## Task 2 — Extract and Inspect Few-Shot Training Samples

### Scenario

You now have a complete training tomogram, five particle-center annotations, and their spherical weak labels. The full volume must be converted into a small training dataset without accidentally using unannotated particles as negative examples.

Your goal is to extract particle-centered positive subvolumes and verified background subvolumes while preserving image-label alignment.

### What You Will Do

In the following code cell, you will:

1. define a cubic training-subvolume size;
2. extract one positive input-label pair around every few-shot center;
3. sample an equal number of verified background subvolumes;
4. reject background candidates that are too close to any synthetic particle;
5. combine the positive and background samples into one training collection;
6. inspect representative input-label pairs;
7. calculate the foreground and background voxel counts;
8. verify that the input and label arrays remain spatially aligned.

### Main Editable Parameters

| Parameter               | Default value | Meaning                                                  |
| ----------------------- | ------------: | -------------------------------------------------------- |
| `SUBVOLUME_SIZE`        |          `24` | Depth, height, and width of each cubic sample            |
| `NUM_BACKGROUND`        |      `N_SHOT` | Number of verified background samples                    |
| `BACKGROUND_CLEARANCE`  |           `2` | Additional empty space required around a background crop |
| `MAX_SAMPLING_ATTEMPTS` |        `5000` | Maximum number of proposed background locations          |
| `NUM_SAMPLES_TO_SHOW`   |           `2` | Number of representative samples to visualize            |

When exploring the results:

* reduce `SUBVOLUME_SIZE` to observe how particle context is lost;
* increase `SUBVOLUME_SIZE` to observe how the background-voxel proportion grows;
* increase `NUM_BACKGROUND` to observe how sample selection affects overall class imbalance.

### Expected Data Structure

The extracted arrays should have the following shapes:

| Array               | Expected shape                       | Contents                            |
| ------------------- | ------------------------------------ | ----------------------------------- |
| `positive_inputs`   | `(N_SHOT, S, S, S)`                  | Particle-centered tomogram crops    |
| `positive_labels`   | `(N_SHOT, S, S, S)`                  | Corresponding spherical-label crops |
| `background_inputs` | `(NUM_BACKGROUND, S, S, S)`          | Verified empty crops                |
| `background_labels` | `(NUM_BACKGROUND, S, S, S)`          | All-zero label crops                |
| `train_inputs`      | `(N_SHOT + NUM_BACKGROUND, S, S, S)` | Combined training inputs            |
| `train_labels`      | `(N_SHOT + NUM_BACKGROUND, S, S, S)` | Combined training labels            |

A channel dimension will be added later when these NumPy arrays are converted into PyTorch tensors.

### Expected Visualizations

The task will produce:

* a representative positive subvolume shown through local XY, XZ, and YZ slices;
* its spherical weak label overlaid on the same views;
* a representative verified background subvolume;
* a comparison of positive and background voxel counts.

The visualizations will use local crops and color maps rather than full black-and-white tomogram slices.

### Checkpoint

At the end of the task, the code should confirm that:

* every input subvolume has a matching label subvolume;
* all extracted samples have shape $(S,S,S)$;
* every positive sample contains a positive label at its local center;
* every background label contains zero positive voxels;
* no verified background crop intersects a known synthetic particle;
* no validation or test volume has been added to the training collection.

The resulting dataset will serve as the **baseline few-shot training set**. In Part 3, the same annotated particles and verified background regions will be used to create additional samples through Volume Infill.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Task 2 turns one full tomogram into aligned 16^3 input-label pairs.
# WriteYourCodeHere — change these values, rerun the cell, and compare sample context and imbalance.
SUBVOLUME_SIZE = 16
NUM_BACKGROUND = N_SHOT
BACKGROUND_CLEARANCE = 2
MAX_SAMPLING_ATTEMPTS = 5000
assert SUBVOLUME_SIZE % 4 == 0, "SUBVOLUME_SIZE must be divisible by 4 for the later U-Net."
half_size = SUBVOLUME_SIZE // 2
task2_rng = np.random.default_rng(SEED + 1)

# WriteYourCodeHere — use identical slices for the image and label so they stay spatially aligned.
def extract_subvolume(volume, center, size):
    center = np.asarray(center, dtype=int); start = center - size // 2; end = start + size
    if np.any(start < 0) or np.any(end > np.asarray(volume.shape)): raise ValueError(f"Crop centered at {tuple(center)} leaves the volume.")
    return volume[tuple(slice(a, b) for a, b in zip(start, end))].copy()

# WriteYourCodeHere — extract one positive pair around every available few-shot annotation.
positive_inputs = np.stack([extract_subvolume(tomogram, center, SUBVOLUME_SIZE) for center in few_shot_centers]).astype(np.float32)
positive_labels = np.stack([extract_subvolume(weak_labels, center, SUBVOLUME_SIZE) for center in few_shot_centers]).astype(np.uint8)

# WriteYourCodeHere — reject any proposed negative crop that approaches a known synthetic particle.
def is_verified_background(center):
    exclusion_half_width = half_size + PARTICLE_RADIUS + BACKGROUND_CLEARANCE
    return not np.any(np.all(np.abs(ground_truth_centers - np.asarray(center)) <= exclusion_half_width, axis=1))

background_centers = []
for _ in range(MAX_SAMPLING_ATTEMPTS):
    candidate = np.array([task2_rng.integers(half_size, axis_size - half_size) for axis_size in tomogram.shape])
    nonoverlapping = all(np.max(np.abs(candidate - previous)) >= SUBVOLUME_SIZE for previous in background_centers)
    if is_verified_background(candidate) and nonoverlapping: background_centers.append(candidate)
    if len(background_centers) == NUM_BACKGROUND: break
if len(background_centers) < NUM_BACKGROUND: raise RuntimeError("Not enough verified background crops. Reduce SUBVOLUME_SIZE, BACKGROUND_CLEARANCE, or NUM_BACKGROUND.")
background_centers = np.asarray(background_centers, dtype=int)

# WriteYourCodeHere — crop the background labels instead of assuming they are zero, then verify them.
background_inputs = np.stack([extract_subvolume(tomogram, center, SUBVOLUME_SIZE) for center in background_centers]).astype(np.float32)
background_labels = np.stack([extract_subvolume(weak_labels, center, SUBVOLUME_SIZE) for center in background_centers]).astype(np.uint8)
assert not background_labels.any(), "A selected background crop contains a weak-label voxel."

# WriteYourCodeHere — combine positive and negative pairs into the baseline training collection.
train_inputs = np.concatenate([positive_inputs, background_inputs], axis=0).astype(np.float32)
train_labels = np.concatenate([positive_labels, background_labels], axis=0).astype(np.uint8)
sample_types = np.array(["positive"] * len(positive_inputs) + ["background"] * len(background_inputs))
training_provenance = np.array(["task1_training_tomogram"] * len(train_inputs))
positive_voxels = int(train_labels.sum()); total_voxels = int(train_labels.size)
positive_fraction = positive_voxels / total_voxels; background_fraction = 1 - positive_fraction
print(f"Positive pairs: {positive_inputs.shape} / {positive_labels.shape}")
print(f"Background pairs: {background_inputs.shape} / {background_labels.shape}")
print(f"Baseline dataset: {train_inputs.shape} / {train_labels.shape}")
print(f"Positive label voxels: {positive_voxels:,}/{total_voxels:,} ({100 * positive_fraction:.3f}%)")

# WriteYourCodeHere — show input, label, overlay, and negative example as four distinct concepts.
mid = half_size; positive_view = positive_inputs[0, mid]; positive_label_view = positive_labels[0, mid]; background_view = background_inputs[0, mid]
display_values = np.concatenate([positive_view.ravel(), background_view.ravel()]); display_min, display_max = np.percentile(display_values, [2, 98])
visible_label = np.ma.masked_where(positive_label_view == 0, positive_label_view); label_cmap = ListedColormap(["#111827", "#22d3ee"])
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6), dpi=140, constrained_layout=True)

intensity_image = axes[0].imshow(positive_view, cmap="magma", origin="lower", interpolation="bilinear", vmin=display_min, vmax=display_max)
axes[0].scatter(mid, mid, marker="+", s=100, color="cyan", linewidths=2); axes[0].set_title("1. Positive input\nNoisy tomogram crop")

axes[1].imshow(positive_label_view, cmap=label_cmap, origin="lower", interpolation="nearest", vmin=0, vmax=1)
axes[1].scatter(mid, mid, marker="+", s=100, color="white", linewidths=2); axes[1].set_title("2. Training label\nCyan voxels = particle")

axes[2].imshow(positive_view, cmap="magma", origin="lower", interpolation="bilinear", vmin=display_min, vmax=display_max)
axes[2].imshow(visible_label, cmap=ListedColormap(["#22d3ee"]), origin="lower", interpolation="nearest", alpha=0.45, vmin=0, vmax=1)
axes[2].scatter(mid, mid, marker="+", s=100, color="cyan", linewidths=2); axes[2].set_title("3. Aligned pair\nInput + weak label")

axes[3].imshow(background_view, cmap="magma", origin="lower", interpolation="bilinear", vmin=display_min, vmax=display_max)
axes[3].set_title("4. Verified background\nLabel = all zeros")

for axis in axes:
    axis.set_xlabel("Local x"); axis.set_ylabel("Local y")
    axis.set_xticks([0, mid, SUBVOLUME_SIZE - 1]); axis.set_yticks([0, mid, SUBVOLUME_SIZE - 1]); axis.set_aspect("equal")

fig.colorbar(intensity_image, ax=[axes[0], axes[2], axes[3]], fraction=0.025, pad=0.02, label="Normalized intensity")
fig.suptitle("How Task 2 constructs positive and background training pairs", fontsize=14)
plt.show()

# WriteYourCodeHere — compare sample-level balance with voxel-level imbalance.
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), dpi=140, constrained_layout=True)
sample_bars = axes[0].bar(["Positive", "Background"], [len(positive_inputs), len(background_inputs)], color=["#06b6d4", "#64748b"])
axes[0].bar_label(sample_bars); axes[0].set_ylabel("Number of samples"); axes[0].set_title("Sample-level balance"); axes[0].spines[["top", "right"]].set_visible(False)

voxel_bars = axes[1].bar(["Positive", "Background"], [100 * positive_fraction, 100 * background_fraction], color=["#f97316", "#64748b"])
axes[1].bar_label(voxel_bars, labels=[f"{100 * positive_fraction:.3f}%", f"{100 * background_fraction:.3f}%"], padding=3)
axes[1].set_ylim(0, 105); axes[1].set_ylabel("Training-label voxels (%)"); axes[1].set_title("Voxel-level imbalance")
axes[1].spines[["top", "right"]].set_visible(False)
plt.show()

# Checkpoint: these assertions explain what a valid baseline dataset must satisfy.
expected_shape = (SUBVOLUME_SIZE,) * 3
assert positive_inputs.shape == positive_labels.shape == (len(few_shot_centers), *expected_shape)
assert background_inputs.shape == background_labels.shape == (NUM_BACKGROUND, *expected_shape)
assert all(positive_labels[i, mid, mid, mid] == 1 for i in range(len(positive_labels)))
assert all(is_verified_background(center) for center in background_centers)
assert train_inputs.shape == train_labels.shape == (len(few_shot_centers) + NUM_BACKGROUND, *expected_shape)
assert set(sample_types) == {"positive", "background"} and set(training_provenance) == {"task1_training_tomogram"}

if "SIGNAL_SUPPORT_RADIUS" in globals():
    for center in few_shot_centers:
        others = ground_truth_centers[np.any(ground_truth_centers != center, axis=1)]
        assert not np.any(np.all(np.abs(others - center) <= half_size + SIGNAL_SUPPORT_RADIUS, axis=1)), "A positive crop contains another synthetic particle."

print("Checkpoint passed: every baseline input-label pair is aligned and every negative crop is verified background.")

## Part 3 — Self-Augmented Volume Infill

Part 2 produced a baseline training set containing five particle-centered samples and five verified background samples. Although the input-label pairs are correctly aligned, the detector can still observe only five annotated particle appearances. Reusing these samples without modification may cause the model to memorize their orientations, positions, and surrounding noise patterns.

[SaSi](https://arxiv.org/abs/2505.19948) introduces Self-Augmented Volume Infill to address two related few-shot problems: limited particle diversity and sparse particle occupancy. The method generates spatially transformed variants of available particles and uses them to increase the number and density of particles in training subvolumes.

This notebook implements a simplified educational version of that idea.

### 3.1 Ordinary Augmentation and Volume Infill

Ordinary augmentation transforms an existing input-label pair:

$$X'=T(X),\qquad Y'=T(Y)$$

The number of particles inside the sample remains unchanged. The particle may appear in a different orientation or position, but the transformed sample still represents the same original training subvolume.

Volume Infill performs an additional step: the transformed particle is inserted into a verified background region to create a new input-label pair.

| Method               | Changes particle appearance? | Creates a new particle-containing region? | Increases particle occupancy? |
| -------------------- | ---------------------------: | ----------------------------------------: | ----------------------------: |
| Rotation or flipping |                          Yes |                                        No |                            No |
| Spatial shifting     |             Changes position |                                        No |                            No |
| Volume Infill        |                          Yes |                                       Yes |                           Yes |

The purpose is not to invent a new biological particle class. It is to reuse the few available annotated particles in new spatial contexts.

### 3.2 Transforming the Particle and Its Label

Let $X_p$ be a positive source subvolume and $Y_p$ its spherical weak label. We apply a spatial transformation $T$ to both arrays:

$$X_p'=T(X_p),\qquad Y_p'=T(Y_p)$$

The transformation set in this tutorial contains:

* 90-degree 3D rotations;
* axis flipping;
* small integer shifts.

Image and label transformations must remain synchronized. If the input particle is shifted but the weak label is not, the generated sample becomes incorrect.

The image transformation may use interpolation when necessary, but the binary label must preserve discrete class values. Therefore, label transformations use nearest-neighbor interpolation or exact array operations.

SaSi uses AugMix-inspired transformation chains and stochastic mixing weights. To keep this notebook focused and CPU-friendly, we will not reproduce its complete Dirichlet- and Beta-weighted mixing process. Instead, each infilled sample will use one randomly selected transformation sequence.

### 3.3 Inserting a Particle into Verified Background

Let $X_b$ be a verified background subvolume and let $A$ be a soft blending mask around the transformed particle. The infilled input is constructed as:

$$X_{\mathrm{infill}}=(1-A)\odot X_b+A\odot X_p'$$

where $\odot$ denotes voxel-wise multiplication.

The corresponding label is:

$$Y_{\mathrm{infill}}=\max(Y_b,Y_p')$$

Because the verified background label $Y_b$ is zero, the new positive label comes from the transformed particle.

A hard cut-and-paste operation can produce an artificial boundary around the inserted patch. The soft mask gradually changes from the particle region to the background region, reducing this edge artifact. However, blending must not become so wide that it removes the particle signal or copies excessive source background.

### 3.4 Reliability Requirements

A generated infilled sample is valid only when:

1. the transformed particle and weak label undergo the same spatial operations;
2. the transformed label remains binary;
3. the particle center remains inside the subvolume;
4. the particle does not wrap around an array boundary during shifting;
5. the blending mask remains between 0 and 1;
6. the original positive and background samples remain unchanged.

These checks are important because an augmented image can look plausible while still containing a misaligned or corrupted training label.

The synthetic result should also not be interpreted as a physically exact CryoET simulation. Volume Infill increases useful training variation, but it does not reproduce every interaction between a macromolecule and its local cellular environment.

---

## Task 3 — Generate SaSi-Inspired Infilled Training Samples

### Scenario

You have only five annotated particle samples, but you also have several verified background subvolumes. Instead of repeatedly training on the same five particle appearances, you will transform the available particles and insert them into new background regions.

### What You Will Do

In the following code cell, you will:

1. select a positive source sample and a verified background sample;
2. construct a soft mask around the source particle;
3. apply random 3D rotations, flips, and shifts;
4. apply the same transformation to the image, weak label, and blending mask;
5. blend the transformed particle into the background;
6. repeat the process to generate multiple infilled samples;
7. combine the infilled samples with the baseline training set;
8. compare particle occupancy before and after Volume Infill.

### Main Editable Parameters

| Parameter            | Default value | Meaning                                      |
| -------------------- | ------------: | -------------------------------------------- |
| `NUM_INFILL_SAMPLES` |          `10` | Number of new particle-containing samples    |
| `MAX_SHIFT`          |           `3` | Maximum integer shift along each axis        |
| `ALLOW_FLIPS`        |        `True` | Whether random axis flips are allowed        |
| `BLENDING_SIGMA`     |         `1.0` | Smoothness of the particle blending boundary |
| `INFILL_SEED`        |    `SEED + 2` | Reproducible random seed for Volume Infill   |

The default shift is deliberately small because each training subvolume is only $16\times16\times16$ voxels. A large shift could move part of the particle outside the sample.

### Expected Data Structure

| Array                    | Expected shape                          | Contents                              |
| ------------------------ | --------------------------------------- | ------------------------------------- |
| `infilled_inputs`        | `(NUM_INFILL_SAMPLES, 16, 16, 16)`      | New particle-containing image samples |
| `infilled_labels`        | `(NUM_INFILL_SAMPLES, 16, 16, 16)`      | Transformed spherical weak labels     |
| `infilled_centers`       | `(NUM_INFILL_SAMPLES, 3)`               | Local particle centers after shifting |
| `augmented_train_inputs` | `(10 + NUM_INFILL_SAMPLES, 16, 16, 16)` | Baseline and infilled inputs          |
| `augmented_train_labels` | `(10 + NUM_INFILL_SAMPLES, 16, 16, 16)` | Baseline and infilled labels          |

The baseline arrays from Part 2 will remain unchanged so that the baseline and Volume Infill models can be compared later.

### Expected Visualizations

The task will show:

* the original positive source sample;
* the transformed particle sample;
* the verified background sample;
* the final infilled sample with its weak-label contour;
* particle-containing sample ratios before and after Volume Infill.

These views will demonstrate that Volume Infill changes both particle appearance and spatial context while preserving image-label alignment.

### Checkpoint

At the end of the task, the code should confirm that:

* all infilled inputs and labels have shape $(16,16,16)$;
* all generated labels remain binary;
* every infilled label contains one connected component;
* the recorded center lies inside its positive weak-label region;
* image, label, and blending-mask transformations remain aligned;
* no shift introduces wrap-around voxels;
* the original baseline arrays have not been modified.

Part 4 will use the baseline and Volume Infill datasets to train two lightweight 3D detectors under the same training settings.


In [ ]:
# Task 3 reuses annotated particles in new orientations, positions, and verified backgrounds.
# WriteYourCodeHere — control how many infilled samples are created and how strongly they are transformed.
NUM_INFILL_SAMPLES = 10
MAX_SHIFT = 3
ALLOW_FLIPS = True
BLENDING_SIGMA = 1.0
INFILL_SEED = SEED + 2
infill_rng = np.random.default_rng(INFILL_SEED)

baseline_inputs_before_infill = train_inputs.copy(); baseline_labels_before_infill = train_labels.copy()
positive_inputs_before_infill = positive_inputs.copy(); background_inputs_before_infill = background_inputs.copy()

# WriteYourCodeHere — find a stable local center from the binary weak label.
def find_label_center(binary_label):
    coordinates = np.argwhere(binary_label > 0)
    if len(coordinates) == 0: raise ValueError("The label contains no positive voxel.")
    centroid = coordinates.mean(axis=0)
    return coordinates[np.argmin(np.sum((coordinates - centroid) ** 2, axis=1))].astype(int)

# WriteYourCodeHere — build a soft mask wider than the weak label so the visible particle is blended smoothly.
def create_soft_particle_mask(shape, center, radius, sigma):
    grids = np.indices(shape, dtype=np.float32)
    distance = np.sqrt(sum((grids[axis] - center[axis]) ** 2 for axis in range(3)))
    sigma = max(float(sigma), 1e-3)
    mask = 1 / (1 + np.exp((distance - radius) / sigma))
    mask[distance > radius + 4 * sigma] = 0
    return (mask / (mask.max() + 1e-8)).astype(np.float32)

# WriteYourCodeHere — apply exactly the same rotations and flips to input, label, and mask.
def apply_orientation_transform(array, rotations, flip_axes):
    transformed = array.copy()
    for axes, steps in rotations: transformed = np.rot90(transformed, k=steps, axes=axes)
    for axis in flip_axes: transformed = np.flip(transformed, axis=axis)
    return transformed.copy()

# WriteYourCodeHere — shift without np.roll so voxels never wrap to the opposite boundary.
def shift_without_wrap(array, shift, fill_value=0):
    shifted = np.full(array.shape, fill_value, dtype=array.dtype); source_slices = []; destination_slices = []
    for axis_size, displacement in zip(array.shape, shift):
        displacement = int(displacement)
        if displacement >= 0:
            source_slices.append(slice(0, axis_size - displacement)); destination_slices.append(slice(displacement, axis_size))
        else:
            source_slices.append(slice(-displacement, axis_size)); destination_slices.append(slice(0, axis_size + displacement))
    shifted[tuple(destination_slices)] = array[tuple(source_slices)]
    return shifted

# WriteYourCodeHere — count 26-connected components without requiring scipy or skimage.
def count_connected_components(binary_volume):
    remaining = set(map(tuple, np.argwhere(binary_volume > 0)))
    offsets = [(dz, dy, dx) for dz in (-1, 0, 1) for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dz, dy, dx) != (0, 0, 0)]
    count = 0

    while remaining:
        count += 1; stack = [remaining.pop()]
        while stack:
            voxel = stack.pop()
            for offset in offsets:
                neighbor = tuple(voxel[i] + offset[i] for i in range(3))
                if neighbor in remaining:
                    remaining.remove(neighbor); stack.append(neighbor)

    return count

# WriteYourCodeHere — generate transformed particles and blend each one into a verified background crop.
infilled_input_list = []; infilled_label_list = []; infilled_center_list = []; infilled_mask_list = []
source_label_voxel_counts = []; infill_transformations = []; example_infill = None
rotation_axis_options = [(0, 1), (0, 2), (1, 2)]

for sample_index in range(NUM_INFILL_SAMPLES):
    source_index = int(infill_rng.integers(len(positive_inputs)))
    background_index = int(infill_rng.integers(len(background_inputs)))

    source_input = positive_inputs[source_index].copy()
    source_label = positive_labels[source_index].copy()
    target_background = background_inputs[background_index].copy()

    source_center = find_label_center(source_label)
    source_mask = create_soft_particle_mask(source_input.shape, source_center, PARTICLE_RADIUS + 1, BLENDING_SIGMA)

    rotations = [(rotation_axis_options[int(infill_rng.integers(3))], int(infill_rng.integers(0, 4))) for _ in range(int(infill_rng.integers(1, 4)))]
    flip_axes = tuple(axis for axis in range(3) if ALLOW_FLIPS and infill_rng.random() < 0.5)
    shift = infill_rng.integers(-MAX_SHIFT, MAX_SHIFT + 1, size=3).astype(int)

    source_background_level = float(np.median(source_input[source_mask < 0.05]))
    target_background_level = float(np.median(target_background))

    oriented_input = apply_orientation_transform(source_input, rotations, flip_axes)
    oriented_label = apply_orientation_transform(source_label, rotations, flip_axes)
    oriented_mask = apply_orientation_transform(source_mask, rotations, flip_axes)

    transformed_input = shift_without_wrap(oriented_input, shift, source_background_level)
    transformed_label = shift_without_wrap(oriented_label, shift, 0).astype(np.uint8)
    transformed_mask = np.clip(shift_without_wrap(oriented_mask, shift, 0.0), 0, 1).astype(np.float32)

    adjusted_source = transformed_input - source_background_level + target_background_level
    infilled_input = ((1 - transformed_mask) * target_background + transformed_mask * adjusted_source).astype(np.float32)
    infilled_center = find_label_center(transformed_label)

    infilled_input_list.append(infilled_input)
    infilled_label_list.append(transformed_label)
    infilled_center_list.append(infilled_center)
    infilled_mask_list.append(transformed_mask)
    source_label_voxel_counts.append(int(source_label.sum()))

    infill_transformations.append({
        "source_index": source_index,
        "background_index": background_index,
        "rotations": rotations,
        "flip_axes": flip_axes,
        "shift": shift.copy()
    })

    if sample_index == 0:
        example_infill = {
            "source_input": source_input,
            "source_center": source_center,
            "transformed_input": adjusted_source,
            "background": target_background,
            "mask": transformed_mask,
            "infilled_input": infilled_input,
            "label": transformed_label,
            "center": infilled_center
        }

infilled_inputs = np.stack(infilled_input_list).astype(np.float32)
infilled_labels = np.stack(infilled_label_list).astype(np.uint8)
infilled_centers = np.stack(infilled_center_list).astype(int)
infilled_blending_masks = np.stack(infilled_mask_list).astype(np.float32)
source_label_voxel_counts = np.asarray(source_label_voxel_counts, dtype=int)

# WriteYourCodeHere — keep baseline data unchanged and append infilled samples to a separate augmented collection.
augmented_train_inputs = np.concatenate([train_inputs, infilled_inputs], axis=0).astype(np.float32)
augmented_train_labels = np.concatenate([train_labels, infilled_labels], axis=0).astype(np.uint8)
augmented_sample_types = np.concatenate([sample_types, np.array(["infilled"] * NUM_INFILL_SAMPLES)])

print(f"Infilled pairs: {infilled_inputs.shape} / {infilled_labels.shape}")
print(f"Augmented dataset: {augmented_train_inputs.shape} / {augmented_train_labels.shape}")

# WriteYourCodeHere — separate source, transformation, mask, background, final input, and final label.
source_center = example_infill["source_center"]; final_center = example_infill["center"]
source_view = example_infill["source_input"][source_center[0]]
transformed_view = example_infill["transformed_input"][final_center[0]]
mask_view = example_infill["mask"][final_center[0]]
background_view = example_infill["background"][final_center[0]]
final_view = example_infill["infilled_input"][final_center[0]]
final_label_view = example_infill["label"][final_center[0]]

intensity_values = np.concatenate([source_view.ravel(), transformed_view.ravel(), background_view.ravel(), final_view.ravel()])
vmin, vmax = np.percentile(intensity_values, [2, 98])
fig, axes = plt.subplots(2, 3, figsize=(10, 7), dpi=140, constrained_layout=True)

image_specs = [
    (source_view, "1. Original source\nAnnotated particle"),
    (transformed_view, "2. Transformed source\nRotation, flip and shift"),
    (mask_view, "3. Soft blending mask\nWhite = copied most strongly"),
    (background_view, "4. Target background\nBefore insertion"),
    (final_view, "5. Final infilled input\nNew positive sample"),
    (final_label_view, "6. Final training label\n1 = particle, 0 = background")
]

for index, (view, title) in enumerate(image_specs):
    axis = axes.flat[index]

    if index in (2, 5):
        axis.imshow(view, cmap="viridis" if index == 2 else label_cmap, origin="lower", interpolation="bilinear" if index == 2 else "nearest", vmin=0, vmax=1)
    else:
        axis.imshow(view, cmap="magma", origin="lower", interpolation="bilinear", vmin=vmin, vmax=vmax)

    center = source_center if index == 0 else final_center

    if index != 2:
        axis.scatter(center[2], center[1], marker="+", s=90, color="cyan" if index != 3 else "white", linewidths=2)

    axis.set_title(title)
    axis.set_xlabel("Local x"); axis.set_ylabel("Local y")
    axis.set_xticks([0, half_size, SUBVOLUME_SIZE - 1])
    axis.set_yticks([0, half_size, SUBVOLUME_SIZE - 1])
    axis.set_aspect("equal")

fig.suptitle("Volume Infill creates a new input-label pair from an existing annotation", fontsize=14)
plt.show()

# WriteYourCodeHere — measure how Volume Infill changes the number of particle-containing samples.
baseline_has_particle = np.any(train_labels.reshape(len(train_labels), -1) > 0, axis=1)
augmented_has_particle = np.any(augmented_train_labels.reshape(len(augmented_train_labels), -1) > 0, axis=1)

counts = [int(baseline_has_particle.sum()), int(augmented_has_particle.sum())]
totals = [len(train_labels), len(augmented_train_labels)]
ratios = [counts[i] / totals[i] for i in range(2)]

fig, ax = plt.subplots(figsize=(5.8, 3.8), dpi=140)
bars = ax.bar(["Baseline", "Volume Infill"], np.array(ratios) * 100, color=["#64748b", "#06b6d4"], width=0.58)
ax.bar_label(bars, labels=[f"{counts[i]}/{totals[i]}\n{100 * ratios[i]:.1f}%" for i in range(2)], padding=4)
ax.set_ylim(0, 110)
ax.set_ylabel("Particle-containing samples (%)")
ax.set_title("Particle occupancy after Volume Infill")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

# Checkpoint: transformations must preserve labels, alignment, masks, and baseline data.
expected_infill_shape = (NUM_INFILL_SAMPLES, SUBVOLUME_SIZE, SUBVOLUME_SIZE, SUBVOLUME_SIZE)
assert infilled_inputs.shape == infilled_labels.shape == expected_infill_shape
assert infilled_centers.shape == (NUM_INFILL_SAMPLES, 3)
assert set(np.unique(infilled_labels)).issubset({0, 1})
assert all(count_connected_components(label) == 1 for label in infilled_labels)
assert all(label[tuple(center)] == 1 for label, center in zip(infilled_labels, infilled_centers))
assert all(int(label.sum()) == source_count for label, source_count in zip(infilled_labels, source_label_voxel_counts))
assert np.isfinite(infilled_inputs).all() and np.isfinite(infilled_blending_masks).all()
assert infilled_blending_masks.min() >= 0 and infilled_blending_masks.max() <= 1
assert all(np.all(mask[label.astype(bool)] > 0.5) for mask, label in zip(infilled_blending_masks, infilled_labels))
assert np.array_equal(train_inputs, baseline_inputs_before_infill)
assert np.array_equal(train_labels, baseline_labels_before_infill)
assert np.array_equal(positive_inputs, positive_inputs_before_infill)
assert np.array_equal(background_inputs, background_inputs_before_infill)
assert augmented_train_inputs.shape == augmented_train_labels.shape
assert len(augmented_train_inputs) == len(train_inputs) + NUM_INFILL_SAMPLES

print("Checkpoint passed: infilled inputs, weak labels, and masks are aligned; baseline data remains unchanged.")

## Part 4 — Training a Lightweight 3D Detector

Parts 2 and 3 created two training collections:

| Model             | Training data                                        |
| ----------------- | ---------------------------------------------------- |
| **Baseline**      | Original positive and verified-background subvolumes |
| **Volume Infill** | Original subvolumes plus infilled samples            |

We will now train two detectors using the same model architecture and optimization settings. The only intended difference is whether Volume Infill samples are included.

### 4.1 Tiny 3D U-Net

The detector receives a normalized subvolume with shape $(1,16,16,16)$, where the first dimension is the image channel. It produces one logit for every input voxel:

$$f_\theta:\mathbb{R}^{1\times16\times16\times16}\rightarrow\mathbb{R}^{1\times16\times16\times16}$$

A sigmoid function converts each output logit $s(\mathbf{v})$ into a particle probability:

$$p(\mathbf{v})=\frac{1}{1+\exp[-s(\mathbf{v})]}$$

The model uses a small encoder-decoder structure:

| Stage      | Main operation                 | Spatial size |
| ---------- | ------------------------------ | -----------: |
| Input      | Normalized subvolume           |       $16^3$ |
| Encoder 1  | 3D convolutions                |       $16^3$ |
| Encoder 2  | Pooling and 3D convolutions    |        $8^3$ |
| Bottleneck | Pooling and 3D convolutions    |        $4^3$ |
| Decoder 2  | Upsampling and skip connection |        $8^3$ |
| Decoder 1  | Upsampling and skip connection |       $16^3$ |
| Output     | $1\times1\times1$ convolution  |       $16^3$ |

Skip connections preserve local particle information that may otherwise be lost during downsampling. The channel counts remain small so that both models can be trained on CPU.

### 4.2 Dice and Focal Loss

Only a small percentage of the training voxels belong to weak particle labels. Ordinary voxel accuracy would therefore be misleading: a model predicting background everywhere could still achieve high accuracy.

Dice loss measures overlap between predicted probabilities $p_i$ and binary labels $y_i$:

$$L_{\mathrm{Dice}}=1-\frac{2\sum_i p_i y_i+\epsilon}{\sum_i p_i+\sum_i y_i+\epsilon}$$

Focal loss reduces the influence of easy background voxels and emphasizes incorrectly predicted voxels:

$$L_{\mathrm{Focal}}=-\frac{1}{M}\sum_i\left[\alpha y_i(1-p_i)^\gamma\log(p_i)+(1-\alpha)(1-y_i)p_i^\gamma\log(1-p_i)\right]$$

The combined objective is:

$$L_{\mathrm{total}}=\lambda_{\mathrm{Dice}}L_{\mathrm{Dice}}+\lambda_{\mathrm{Focal}}L_{\mathrm{Focal}}$$

[SaSi](https://arxiv.org/abs/2505.19948) also combines Dice and Focal losses for few-shot CryoET particle detection. This tutorial uses smaller, educational settings rather than reproducing the paper’s empirical loss weights and large-scale training schedule.

### 4.3 Fair Training and Independent Validation

The augmented dataset contains more samples than the baseline dataset. Training both models for the same number of epochs would therefore give the Volume Infill model more optimizer updates. To make the comparison fair, both models will instead use the same fixed number of training steps, batch size, initialization, optimizer, and learning rate.

An independent synthetic validation tomogram will be generated using a different random seed. Its particles are not used to build either training dataset. Validation loss will be used to retain the best model state, but final performance will be measured on another unseen test tomogram.

Training loss alone is not evidence of successful particle detection. A model may fit the spherical weak labels without generalizing to new particle locations. We will therefore inspect both loss curves and validation probability maps.

---

## Task 4 — Train the Baseline and Volume Infill Detectors

### Scenario

You have two training datasets and want to determine whether Volume Infill improves learning. To isolate its effect, both experiments must use the same detector and training conditions.

### What You Will Do

In the following code cell, you will:

1. convert NumPy subvolumes into PyTorch tensors with shape $(N,1,16,16,16)$;
2. define a Tiny 3D U-Net;
3. implement Dice loss, Focal loss, and their combined objective;
4. generate an independent validation tomogram and validation subvolumes;
5. initialize the baseline and Volume Infill models identically;
6. train both models for the same number of optimizer steps;
7. retain the best validation state for each model;
8. compare training and validation loss curves;
9. inspect representative validation probability maps.

### Main Editable Parameters

| Parameter         | Default value | Meaning                                       |
| ----------------- | ------------: | --------------------------------------------- |
| `BASE_CHANNELS`   |           `4` | Number of channels in the first encoder stage |
| `BATCH_SIZE`      |           `4` | Number of subvolumes per optimization step    |
| `TRAIN_STEPS`     |         `200` | Optimizer updates for each model              |
| `LEARNING_RATE`   |        `1e-3` | Adam learning rate                            |
| `DICE_WEIGHT`     |         `1.0` | Contribution of Dice loss                     |
| `FOCAL_WEIGHT`    |         `1.0` | Contribution of Focal loss                    |
| `FOCAL_ALPHA`     |        `0.75` | Relative emphasis on positive voxels          |
| `FOCAL_GAMMA`     |         `2.0` | Focusing strength for difficult voxels        |
| `VALIDATION_SEED` |  `SEED + 100` | Seed for the independent validation tomogram  |

### Expected Outputs

The code will create `baseline_model`, `infill_model`, `baseline_history`, `infill_history`, `validation_tomogram`, `validation_labels`, and `validation_centers`. The loss histories will record training and validation loss at regular intervals.

The expected visualizations are:

* baseline and Volume Infill training-loss curves;
* validation-loss curves;
* validation input, weak label, baseline probability, and Volume Infill probability for the same subvolume.

### Checkpoint

The code should confirm that both models:

* begin from identical weights;
* receive tensors with shape $(N,1,16,16,16)$;
* produce outputs with the same shape as their labels;
* use the same number of optimizer steps;
* produce finite losses and probabilities between 0 and 1;
* never use the future test tomogram during training or model selection.

A lower validation loss is encouraging, but the final comparison must use particle coordinates rather than segmentation loss.


In [ ]:
import copy

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ModuleNotFoundError as error:
    raise ModuleNotFoundError("Task 4 requires PyTorch. Select the notebook environment that contains torch, then rerun this cell.") from error

# Task 4 trains two identical detectors; training data is the only intended difference.
# WriteYourCodeHere — adjust training capacity and optimization settings, then compare validation behavior.
BASE_CHANNELS = 4
BATCH_SIZE = 4
TRAIN_STEPS = 200
LEARNING_RATE = 1e-3
DICE_WEIGHT = 1.0
FOCAL_WEIGHT = 1.0
FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0
VALIDATION_SEED = SEED + 100
VALIDATION_SAMPLES_PER_CLASS = 8
VALIDATION_INTERVAL = 10
MODEL_SEED = SEED + 3
TRAINING_SEED = SEED + 4
DEVICE = torch.device("cpu")

# WriteYourCodeHere — generate an independent volume with new centers, particle appearances, and noise.
def generate_independent_tomogram(seed, num_particles=NUM_PARTICLES):
    local_rng = np.random.default_rng(seed); centers = []

    for _ in range(MAX_CENTER_ATTEMPTS):
        candidate = np.array([local_rng.integers(CENTER_MARGIN, axis_size - CENTER_MARGIN) for axis_size in VOLUME_SHAPE], dtype=int)

        if all(np.linalg.norm(candidate - previous) >= MIN_PARTICLE_DISTANCE for previous in centers):
            centers.append(candidate)

        if len(centers) == num_particles:
            break

    if len(centers) < num_particles:
        raise RuntimeError("Could not generate the independent validation centers.")

    centers = np.asarray(centers, dtype=int)
    clean = np.zeros(VOLUME_SHAPE, dtype=np.float32)
    support = SIGNAL_SUPPORT_RADIUS

    for center in centers:
        particle = make_particle_template(local_rng, support) * local_rng.uniform(1.8, 2.4)
        z, y, x = center
        clean[z - support:z + support + 1, y - support:y + support + 1, x - support:x + support + 1] += particle

    white = local_rng.normal(size=VOLUME_SHAPE).astype(np.float32)
    fz = np.fft.fftfreq(VOLUME_SHAPE[0])[:, None, None]
    fy = np.fft.fftfreq(VOLUME_SHAPE[1])[None, :, None]
    fx = np.fft.fftfreq(VOLUME_SHAPE[2])[None, None, :]
    frequency_filter = np.exp(-((fz / 0.10) ** 2 + (fy / 0.16) ** 2 + (fx / 0.16) ** 2))

    correlated = np.fft.ifftn(np.fft.fftn(white) * frequency_filter).real.astype(np.float32)
    correlated = (correlated - correlated.mean()) / (correlated.std() + 1e-8)

    volume = clean + NOISE_STRENGTH * correlated + 0.12 * local_rng.normal(size=VOLUME_SHAPE).astype(np.float32)
    volume = ((volume - volume.mean()) / (volume.std() + 1e-8)).astype(np.float32)

    labels = np.zeros(VOLUME_SHAPE, dtype=np.uint8)
    zg, yg, xg = np.ogrid[:VOLUME_SHAPE[0], :VOLUME_SHAPE[1], :VOLUME_SHAPE[2]]

    for z, y, x in centers:
        labels[(zg - z) ** 2 + (yg - y) ** 2 + (xg - x) ** 2 <= WEAK_LABEL_RADIUS ** 2] = 1

    return volume, centers, labels

validation_tomogram, validation_centers, validation_labels = generate_independent_tomogram(VALIDATION_SEED)
assert VALIDATION_SEED != SEED, "Validation must use a seed different from the training tomogram."

# WriteYourCodeHere — build a balanced validation crop set without adding it to either training array.
validation_rng = np.random.default_rng(VALIDATION_SEED + 1)
num_validation_positive = min(VALIDATION_SAMPLES_PER_CLASS, len(validation_centers))
validation_positive_centers = validation_centers[:num_validation_positive]

validation_positive_inputs = np.stack([extract_subvolume(validation_tomogram, center, SUBVOLUME_SIZE) for center in validation_positive_centers])
validation_positive_labels = np.stack([extract_subvolume(validation_labels, center, SUBVOLUME_SIZE) for center in validation_positive_centers])

def is_validation_background(center):
    exclusion = half_size + PARTICLE_RADIUS + BACKGROUND_CLEARANCE
    return not np.any(np.all(np.abs(validation_centers - np.asarray(center)) <= exclusion, axis=1))

validation_background_centers = []
used_validation_centers = set()

for _ in range(MAX_SAMPLING_ATTEMPTS):
    candidate = np.array([validation_rng.integers(half_size, axis_size - half_size) for axis_size in validation_tomogram.shape])
    key = tuple(candidate)

    if is_validation_background(candidate) and key not in used_validation_centers:
        validation_background_centers.append(candidate)
        used_validation_centers.add(key)

    if len(validation_background_centers) == VALIDATION_SAMPLES_PER_CLASS:
        break

if len(validation_background_centers) < VALIDATION_SAMPLES_PER_CLASS:
    raise RuntimeError("Not enough independent validation background crops.")

validation_background_centers = np.asarray(validation_background_centers, dtype=int)
validation_background_inputs = np.stack([extract_subvolume(validation_tomogram, center, SUBVOLUME_SIZE) for center in validation_background_centers])
validation_background_labels = np.stack([extract_subvolume(validation_labels, center, SUBVOLUME_SIZE) for center in validation_background_centers])
assert not validation_background_labels.any()

validation_inputs = np.concatenate([validation_positive_inputs, validation_background_inputs]).astype(np.float32)
validation_crop_labels = np.concatenate([validation_positive_labels, validation_background_labels]).astype(np.uint8)
validation_provenance = np.array(["independent_validation_tomogram"] * len(validation_inputs))

# WriteYourCodeHere — add the channel dimension required by Conv3d: (N,D,H,W) -> (N,1,D,H,W).
def to_5d_tensor(array):
    return torch.from_numpy(array.astype(np.float32))[:, None]

baseline_input_tensor = to_5d_tensor(train_inputs)
baseline_label_tensor = to_5d_tensor(train_labels)
infill_input_tensor = to_5d_tensor(augmented_train_inputs)
infill_label_tensor = to_5d_tensor(augmented_train_labels)
validation_input_tensor = to_5d_tensor(validation_inputs)
validation_label_tensor = to_5d_tensor(validation_crop_labels)

print(f"Baseline tensors: {tuple(baseline_input_tensor.shape)} / {tuple(baseline_label_tensor.shape)}")
print(f"Infill tensors: {tuple(infill_input_tensor.shape)} / {tuple(infill_label_tensor.shape)}")
print(f"Validation tensors: {tuple(validation_input_tensor.shape)} / {tuple(validation_label_tensor.shape)}")

# WriteYourCodeHere — define a small U-Net; GroupNorm is stable with a batch size of four.
class ConvBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, 3, padding=1),
            nn.GroupNorm(1, out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(1, out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.layers(x)

class TinyUNet3D(nn.Module):
    def __init__(self, base_channels=4):
        super().__init__()
        b = base_channels

        self.encoder1 = ConvBlock3D(1, b)
        self.pool1 = nn.MaxPool3d(2)
        self.encoder2 = ConvBlock3D(b, 2 * b)
        self.pool2 = nn.MaxPool3d(2)

        self.bottleneck = ConvBlock3D(2 * b, 4 * b)

        self.up2 = nn.ConvTranspose3d(4 * b, 2 * b, 2, stride=2)
        self.decoder2 = ConvBlock3D(4 * b, 2 * b)

        self.up1 = nn.ConvTranspose3d(2 * b, b, 2, stride=2)
        self.decoder1 = ConvBlock3D(2 * b, b)

        self.output = nn.Conv3d(b, 1, 1)

    def forward(self, x):
        encoder1 = self.encoder1(x)
        encoder2 = self.encoder2(self.pool1(encoder1))
        bottleneck = self.bottleneck(self.pool2(encoder2))

        decoder2 = self.decoder2(torch.cat([self.up2(bottleneck), encoder2], dim=1))
        decoder1 = self.decoder1(torch.cat([self.up1(decoder2), encoder1], dim=1))

        return self.output(decoder1)

# WriteYourCodeHere — combine overlap-sensitive Dice loss with imbalance-aware focal loss.
def dice_loss(logits, targets, epsilon=1e-6):
    probabilities = torch.sigmoid(logits)
    dimensions = (1, 2, 3, 4)
    numerator = 2 * (probabilities * targets).sum(dim=dimensions) + epsilon
    denominator = probabilities.sum(dim=dimensions) + targets.sum(dim=dimensions) + epsilon
    return (1 - numerator / denominator).mean()

def focal_loss(logits, targets, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
    probabilities = torch.sigmoid(logits)
    cross_entropy = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    correct_probability = probabilities * targets + (1 - probabilities) * (1 - targets)
    alpha_factor = alpha * targets + (1 - alpha) * (1 - targets)
    return (alpha_factor * (1 - correct_probability).pow(gamma) * cross_entropy).mean()

def combined_loss(logits, targets):
    return DICE_WEIGHT * dice_loss(logits, targets) + FOCAL_WEIGHT * focal_loss(logits, targets)

# WriteYourCodeHere — evaluate validation data in small batches without gradient updates.
def evaluate_dataset_loss(model, inputs, labels, batch_size=BATCH_SIZE):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for start in range(0, len(inputs), batch_size):
            batch_inputs = inputs[start:start + batch_size].to(DEVICE)
            batch_labels = labels[start:start + batch_size].to(DEVICE)
            total_loss += combined_loss(model(batch_inputs), batch_labels).item() * len(batch_inputs)

    return total_loss / len(inputs)

# WriteYourCodeHere — train for a fixed number of optimizer steps and retain the lowest-validation-loss state.
def train_detector(training_inputs, training_labels, initial_state, sampling_seed):
    model = TinyUNet3D(BASE_CHANNELS).to(DEVICE)
    model.load_state_dict(copy.deepcopy(initial_state))

    initial_parameters = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    generator = torch.Generator().manual_seed(sampling_seed)

    best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
    best_validation_loss = evaluate_dataset_loss(model, validation_input_tensor, validation_label_tensor)
    best_step = 0

    history = {"step": [], "training_loss": [], "validation_loss": []}
    recent_training_losses = []
    update_count = 0
    model.train()

    for step in range(1, TRAIN_STEPS + 1):
        indices = torch.randint(0, len(training_inputs), (BATCH_SIZE,), generator=generator)
        batch_inputs = training_inputs[indices].to(DEVICE)
        batch_labels = training_labels[indices].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_inputs)
        loss = combined_loss(logits, batch_labels)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite training loss at step {step}.")

        loss.backward()
        optimizer.step()

        update_count += 1
        recent_training_losses.append(float(loss.detach().cpu()))

        if step % VALIDATION_INTERVAL == 0 or step == TRAIN_STEPS:
            validation_loss_value = evaluate_dataset_loss(model, validation_input_tensor, validation_label_tensor)

            history["step"].append(step)
            history["training_loss"].append(float(np.mean(recent_training_losses)))
            history["validation_loss"].append(validation_loss_value)
            recent_training_losses = []

            if validation_loss_value < best_validation_loss:
                best_validation_loss = validation_loss_value
                best_step = step
                best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

            model.train()

    model.load_state_dict(best_state)
    model.eval()

    return model, history, best_step, best_validation_loss, initial_parameters, update_count

# WriteYourCodeHere — create one shared initialization, then train both experiments independently.
torch.manual_seed(MODEL_SEED)
initial_model = TinyUNet3D(BASE_CHANNELS)
shared_initial_state = {name: value.detach().cpu().clone() for name, value in initial_model.state_dict().items()}

baseline_model, baseline_history, baseline_best_step, baseline_best_validation_loss, baseline_initial_parameters, baseline_update_count = train_detector(
    baseline_input_tensor,
    baseline_label_tensor,
    shared_initial_state,
    TRAINING_SEED
)

infill_model, infill_history, infill_best_step, infill_best_validation_loss, infill_initial_parameters, infill_update_count = train_detector(
    infill_input_tensor,
    infill_label_tensor,
    shared_initial_state,
    TRAINING_SEED
)

print(f"Baseline best validation loss: {baseline_best_validation_loss:.4f} at step {baseline_best_step}")
print(f"Volume Infill best validation loss: {infill_best_validation_loss:.4f} at step {infill_best_step}")

# WriteYourCodeHere — compare loss against optimizer step, not epoch.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), dpi=140, constrained_layout=True)

axes[0].plot(baseline_history["step"], baseline_history["training_loss"], label="Baseline", color="#64748b")
axes[0].plot(infill_history["step"], infill_history["training_loss"], label="Volume Infill", color="#06b6d4")
axes[0].set_title("Training loss")
axes[0].set_xlabel("Optimizer step")
axes[0].set_ylabel("Dice + Focal loss")
axes[0].legend()

axes[1].plot(baseline_history["step"], baseline_history["validation_loss"], label="Baseline", color="#64748b")
axes[1].plot(infill_history["step"], infill_history["validation_loss"], label="Volume Infill", color="#06b6d4")
axes[1].set_title("Independent validation loss")
axes[1].set_xlabel("Optimizer step")
axes[1].set_ylabel("Dice + Focal loss")
axes[1].legend()

for axis in axes:
    axis.grid(alpha=0.2)
    axis.spines[["top", "right"]].set_visible(False)

plt.show()

# WriteYourCodeHere — inspect both probability maps on exactly the same unseen validation crop.
example_index = 0
example_tensor = validation_input_tensor[example_index:example_index + 1].to(DEVICE)

with torch.no_grad():
    baseline_probability = torch.sigmoid(baseline_model(example_tensor))[0, 0].cpu().numpy()
    infill_probability = torch.sigmoid(infill_model(example_tensor))[0, 0].cpu().numpy()

validation_view = validation_inputs[example_index, half_size]
validation_label_view = validation_crop_labels[example_index, half_size]
baseline_probability_view = baseline_probability[half_size]
infill_probability_view = infill_probability[half_size]
input_min, input_max = np.percentile(validation_view, [2, 98])

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5), dpi=140, constrained_layout=True)

axes[0].imshow(validation_view, cmap="magma", origin="lower", interpolation="bilinear", vmin=input_min, vmax=input_max)
axes[0].set_title("Validation input")

axes[1].imshow(validation_label_view, cmap=label_cmap, origin="lower", interpolation="nearest", vmin=0, vmax=1)
axes[1].set_title("Validation weak label")

probability_image = axes[2].imshow(baseline_probability_view, cmap="viridis", origin="lower", interpolation="bilinear", vmin=0, vmax=1)
axes[2].set_title("Baseline probability")

axes[3].imshow(infill_probability_view, cmap="viridis", origin="lower", interpolation="bilinear", vmin=0, vmax=1)
axes[3].set_title("Volume Infill probability")

for axis in axes:
    axis.set_xlabel("Local x")
    axis.set_ylabel("Local y")
    axis.set_xticks([0, half_size, SUBVOLUME_SIZE - 1])
    axis.set_yticks([0, half_size, SUBVOLUME_SIZE - 1])
    axis.set_aspect("equal")

fig.colorbar(probability_image, ax=[axes[2], axes[3]], fraction=0.025, pad=0.02, label="Predicted particle probability")
fig.suptitle("Same independent validation sample, same display scale", fontsize=14)
plt.show()

# Checkpoint: fairness, tensor shapes, finite losses, probabilities, and data separation.
assert all(torch.equal(baseline_initial_parameters[name], infill_initial_parameters[name]) for name in baseline_initial_parameters)
assert baseline_input_tensor.ndim == infill_input_tensor.ndim == validation_input_tensor.ndim == 5
assert baseline_input_tensor.shape[1] == infill_input_tensor.shape[1] == validation_input_tensor.shape[1] == 1

with torch.no_grad():
    baseline_shape = baseline_model(baseline_input_tensor[:1].to(DEVICE)).shape
    infill_shape = infill_model(infill_input_tensor[:1].to(DEVICE)).shape

assert baseline_shape == baseline_label_tensor[:1].shape
assert infill_shape == infill_label_tensor[:1].shape
assert baseline_update_count == infill_update_count == TRAIN_STEPS
assert np.isfinite(baseline_history["training_loss"]).all()
assert np.isfinite(infill_history["training_loss"]).all()
assert np.isfinite(baseline_history["validation_loss"]).all()
assert np.isfinite(infill_history["validation_loss"]).all()
assert baseline_probability.min() >= 0 and baseline_probability.max() <= 1
assert infill_probability.min() >= 0 and infill_probability.max() <= 1
assert set(training_provenance) == {"task1_training_tomogram"}
assert set(validation_provenance) == {"independent_validation_tomogram"}
assert not np.shares_memory(train_inputs, validation_inputs)
assert not np.shares_memory(augmented_train_inputs, validation_inputs)

print("Checkpoint passed: both models used identical settings and initialization; validation remained independent.")

## Part 5 — Whole-Volume Inference and Particle Localization

The models trained in Part 4 accept $16^3$ subvolumes, but the final objective is to detect particles throughout an entire tomogram. We therefore need to scan the volume, combine overlapping predictions, and convert the resulting probability map into particle coordinates.

### 5.1 Overlapping Sliding-Window Inference

A window of size $S=16$ moves through the tomogram with stride $s=8$. Because $s<S$, neighboring windows overlap. Every window is processed independently by the detector and returned to its original spatial location.

Let $p_k(\mathbf{v})$ be the probability predicted for voxel $\mathbf{v}$ by window $k$. If $\mathcal{K}(\mathbf{v})$ is the set of windows covering that voxel, the combined probability is:

$$P(\mathbf{v})=\frac{1}{|\mathcal{K}(\mathbf{v})|}\sum_{k\in\mathcal{K}(\mathbf{v})}p_k(\mathbf{v})$$

Averaging overlapping predictions reduces discontinuities near window boundaries. This follows the general overlap-tile motivation used by [DeepETPicker](https://www.nature.com/articles/s41467-024-46041-0), although this notebook uses simple uniform averaging.

An independently generated test tomogram will be used for inference. Its ground-truth centers are retained for Part 6 but are never given to either detector.

### 5.2 From Probability Volume to Binary Regions

The full probability volume is converted into a binary prediction using a threshold $\tau$:

$$
B(\mathbf{v})=
\begin{cases}
1, & P(\mathbf{v}) \geq \tau,\\
0, & P(\mathbf{v}) < \tau.
\end{cases}
$$

A low threshold generally produces more detected regions, increasing recall but also increasing false positives. A high threshold removes uncertain regions, which may improve precision while missing faint particles.

Part 5 uses a provisional threshold of $0.5$ to demonstrate localization. The final threshold will be selected using the validation tomogram in Part 6.

### 5.3 Connected Components and Particle Centers

A binary particle region may contain many positive voxels but should produce only one coordinate. We therefore apply 3D connected-component analysis using 26-connectivity, where voxels touching through faces, edges, or corners can belong to the same component.

Very small components are removed as likely noise. For each remaining component $C_j$, its probability-weighted center is:

$$\hat{\mathbf{c}}*j=\frac{\sum*{\mathbf{v}\in C_j}P(\mathbf{v})\mathbf{v}}{\sum_{\mathbf{v}\in C_j}P(\mathbf{v})}$$

The resulting coordinate is stored in $(z,y,x)$ order. Connected-component localization is the primary post-processing method used in SaSi’s few-shot experiments, while DeepETPicker uses its own MP-NMS procedure. This tutorial uses connected components because the relationship between voxel regions and particle coordinates is easier to inspect.

---

## Task 5 — Predict Full Probability Volumes and Extract Particle Centers

### Scenario

The two trained models can process small subvolumes but have not yet been applied to a complete unseen tomogram. Your goal is to reconstruct full probability maps and obtain provisional particle coordinates.

### What You Will Do

In the following code cell, you will:

1. generate an independent synthetic test tomogram;
2. scan the validation and test tomograms using overlapping $16^3$ windows;
3. process windows in small batches for efficient inference;
4. average predictions in overlapping regions;
5. verify that every voxel receives at least one prediction;
6. threshold each full probability volume;
7. apply 26-connected-component analysis;
8. remove very small components;
9. calculate probability-weighted particle centers;
10. visualize probability maps and provisional detections.

### Main Editable Parameters

| Parameter               | Default value | Meaning                                |
| ----------------------- | ------------: | -------------------------------------- |
| `WINDOW_SIZE`           |          `16` | Sliding-window size                    |
| `INFERENCE_STRIDE`      |           `8` | Distance between neighboring windows   |
| `INFERENCE_BATCH_SIZE`  |          `16` | Windows processed together             |
| `PROVISIONAL_THRESHOLD` |         `0.5` | Temporary probability threshold        |
| `MIN_COMPONENT_VOXELS`  |           `5` | Minimum accepted component size        |
| `TEST_SEED`             |  `SEED + 200` | Seed for the independent test tomogram |

### Expected Outputs

The main probability arrays are:

| Array                       | Contents                                            |
| --------------------------- | --------------------------------------------------- |
| `baseline_val_probability`  | Baseline prediction on the validation tomogram      |
| `infill_val_probability`    | Volume Infill prediction on the validation tomogram |
| `baseline_test_probability` | Baseline prediction on the test tomogram            |
| `infill_test_probability`   | Volume Infill prediction on the test tomogram       |

The code will also create provisional `baseline_predicted_centers` and `infill_predicted_centers` from the test probability volumes.

The expected visualizations are:

* an input test slice;
* baseline and Volume Infill probability slices;
* thresholded particle regions;
* provisional particle centers displayed in 3D.

### Checkpoint

The code should confirm that:

* every probability volume has the same shape as its tomogram;
* all probabilities remain in $[0,1]$;
* the overlap-count volume contains no zeros;
* all predicted coordinates remain inside the tomogram;
* every retained coordinate corresponds to one connected component;
* inference does not access ground-truth centers.

These detections remain provisional because the threshold has not yet been selected on validation data.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

# SciPy can accelerate 3D connected-component analysis. If it is unavailable,
# the code automatically uses the NumPy implementation defined below.
try:
    from scipy.ndimage import label as scipy_label, center_of_mass as scipy_center_of_mass
    SCIPY_CONNECTED_COMPONENTS = True
except ModuleNotFoundError:
    SCIPY_CONNECTED_COMPONENTS = False

# ============================================================
# Task 5 — Whole-volume inference and particle localization
# ============================================================

# Task 5 applies the trained patch detectors to complete tomograms.
# No model weights are updated during this task.

# WriteYourCodeHere — adjust inference density and provisional post-processing settings.
WINDOW_SIZE = 16
INFERENCE_STRIDE = 8
INFERENCE_BATCH_SIZE = 16
PROVISIONAL_THRESHOLD = 0.5
MIN_COMPONENT_VOXELS = 5
TEST_SEED = SEED + 200

assert WINDOW_SIZE == SUBVOLUME_SIZE, "Inference windows must match the training-subvolume size."
assert 0 < INFERENCE_STRIDE <= WINDOW_SIZE, "INFERENCE_STRIDE must be between 1 and WINDOW_SIZE."
assert TEST_SEED not in (SEED, VALIDATION_SEED), "Training, validation, and test seeds must be different."


# ============================================================
# 1. Generate an independent test tomogram
# ============================================================

# WriteYourCodeHere — generate a test volume with new particle locations,
# appearances, and noise. Ground truth is retained only for Task 6.
test_tomogram, test_centers, test_labels = generate_independent_tomogram(TEST_SEED)
test_provenance = "independent_test_tomogram"

print(f"Connected-component implementation: {'SciPy' if SCIPY_CONNECTED_COMPONENTS else 'NumPy fallback'}")
print(f"Validation volume: {validation_tomogram.shape}, seed={VALIDATION_SEED}")
print(f"Test volume: {test_tomogram.shape}, seed={TEST_SEED}")


# ============================================================
# 2. Define overlapping sliding-window inference
# ============================================================

# WriteYourCodeHere — include the last legal window position so that boundary
# voxels are never left without a prediction.
def sliding_window_starts(axis_size, window_size, stride):
    if window_size > axis_size:
        raise ValueError("WINDOW_SIZE cannot exceed a tomogram axis.")

    starts = list(range(0, axis_size - window_size + 1, stride))

    if starts[-1] != axis_size - window_size:
        starts.append(axis_size - window_size)

    return starts


# WriteYourCodeHere — predict overlapping windows and average their
# probabilities at the corresponding global coordinates.
def predict_probability_volume(model, volume, window_size=WINDOW_SIZE, stride=INFERENCE_STRIDE, batch_size=INFERENCE_BATCH_SIZE):
    z_starts = sliding_window_starts(volume.shape[0], window_size, stride)
    y_starts = sliding_window_starts(volume.shape[1], window_size, stride)
    x_starts = sliding_window_starts(volume.shape[2], window_size, stride)
    positions = [(z, y, x) for z in z_starts for y in y_starts for x in x_starts]

    probability_sum = np.zeros(volume.shape, dtype=np.float32)
    overlap_count = np.zeros(volume.shape, dtype=np.uint16)
    model.eval()

    with torch.no_grad():
        for start in range(0, len(positions), batch_size):
            batch_positions = positions[start:start + batch_size]

            windows = np.stack([
                volume[z:z + window_size, y:y + window_size, x:x + window_size]
                for z, y, x in batch_positions
            ]).astype(np.float32)

            batch_tensor = torch.from_numpy(windows)[:, None].to(DEVICE)
            batch_probabilities = torch.sigmoid(model(batch_tensor))[:, 0].cpu().numpy()

            for probability, (z, y, x) in zip(batch_probabilities, batch_positions):
                probability_sum[z:z + window_size, y:y + window_size, x:x + window_size] += probability
                overlap_count[z:z + window_size, y:y + window_size, x:x + window_size] += 1

    if np.any(overlap_count == 0):
        raise RuntimeError("Sliding-window inference left uncovered voxels.")

    probability_volume = probability_sum / overlap_count.astype(np.float32)
    return probability_volume.astype(np.float32), overlap_count, positions


# ============================================================
# 3. Define connected-component localization
# ============================================================

# WriteYourCodeHere — use a NumPy flood fill only when SciPy is unavailable.
def numpy_connected_components(binary_volume):
    visited = np.zeros(binary_volume.shape, dtype=bool)
    components = []

    offsets = [
        (dz, dy, dx)
        for dz in (-1, 0, 1)
        for dy in (-1, 0, 1)
        for dx in (-1, 0, 1)
        if (dz, dy, dx) != (0, 0, 0)
    ]

    for seed in np.argwhere(binary_volume):
        seed = tuple(seed)

        if visited[seed]:
            continue

        visited[seed] = True
        queue = deque([seed])
        coordinates = []

        while queue:
            voxel = queue.popleft()
            coordinates.append(voxel)

            for offset in offsets:
                neighbor = tuple(voxel[axis] + offset[axis] for axis in range(3))
                inside_volume = all(0 <= neighbor[axis] < binary_volume.shape[axis] for axis in range(3))

                if inside_volume and binary_volume[neighbor] and not visited[neighbor]:
                    visited[neighbor] = True
                    queue.append(neighbor)

        components.append(np.asarray(coordinates, dtype=int))

    return components


# WriteYourCodeHere — threshold probabilities, reject very small components,
# and calculate one probability-weighted center for every retained component.
def extract_particle_centers(probability_volume, threshold=PROVISIONAL_THRESHOLD, min_component_voxels=MIN_COMPONENT_VOXELS):
    binary_volume = probability_volume >= threshold

    if SCIPY_CONNECTED_COMPONENTS:
        labeled_volume, component_count = scipy_label(binary_volume, structure=np.ones((3, 3, 3), dtype=np.uint8))
        component_sizes = np.bincount(labeled_volume.ravel(), minlength=component_count + 1)[1:]
        accepted_ids = np.flatnonzero(component_sizes >= min_component_voxels) + 1

        if len(accepted_ids) == 0:
            return np.empty((0, 3), dtype=np.float32), np.zeros_like(binary_volume), np.empty(0, dtype=int)

        centers = np.asarray(
            scipy_center_of_mass(probability_volume, labeled_volume, accepted_ids),
            dtype=np.float32
        ).reshape(-1, 3)

        accepted_mask = np.isin(labeled_volume, accepted_ids)
        accepted_sizes = component_sizes[accepted_ids - 1].astype(int)
        return centers, accepted_mask, accepted_sizes

    accepted_components = [
        component
        for component in numpy_connected_components(binary_volume)
        if len(component) >= min_component_voxels
    ]

    centers = []
    accepted_mask = np.zeros_like(binary_volume)

    for component in accepted_components:
        component_index = tuple(component.T)
        weights = probability_volume[component_index]
        weight_sum = float(weights.sum())

        if weight_sum > 0:
            center = (component * weights[:, None]).sum(axis=0) / weight_sum
        else:
            center = component.mean(axis=0)

        centers.append(center)
        accepted_mask[component_index] = True

    centers = np.asarray(centers, dtype=np.float32).reshape(-1, 3)
    accepted_sizes = np.asarray([len(component) for component in accepted_components], dtype=int)
    return centers, accepted_mask, accepted_sizes


# ============================================================
# 4. Predict complete validation and test probability volumes
# ============================================================

# WriteYourCodeHere — run exactly the same inference process for both models.
print("\nRunning baseline inference on the validation volume...")
baseline_val_probability, baseline_val_overlap_count, validation_window_positions = predict_probability_volume(
    baseline_model,
    validation_tomogram
)

print("Running Volume Infill inference on the validation volume...")
infill_val_probability, infill_val_overlap_count, _ = predict_probability_volume(
    infill_model,
    validation_tomogram
)

print("Running baseline inference on the test volume...")
baseline_test_probability, baseline_test_overlap_count, test_window_positions = predict_probability_volume(
    baseline_model,
    test_tomogram
)

print("Running Volume Infill inference on the test volume...")
infill_test_probability, infill_test_overlap_count, _ = predict_probability_volume(
    infill_model,
    test_tomogram
)


# ============================================================
# 5. Inspect probability distributions
# ============================================================

# WriteYourCodeHere — inspect whether the model produces meaningful spatial
# probability variation before choosing a detection threshold.
def print_probability_diagnostics(name, probability_volume):
    percentiles = np.percentile(probability_volume, [1, 50, 95, 99, 99.9])

    print(f"\n{name}:")
    print(f"  Minimum: {probability_volume.min():.4f}")
    print(f"  Mean: {probability_volume.mean():.4f}")
    print(f"  Standard deviation: {probability_volume.std():.4f}")
    print(f"  1st percentile: {percentiles[0]:.4f}")
    print(f"  50th percentile: {percentiles[1]:.4f}")
    print(f"  95th percentile: {percentiles[2]:.4f}")
    print(f"  99th percentile: {percentiles[3]:.4f}")
    print(f"  99.9th percentile: {percentiles[4]:.4f}")
    print(f"  Maximum: {probability_volume.max():.4f}")


probability_diagnostics = [
    ("Baseline validation", baseline_val_probability),
    ("Volume Infill validation", infill_val_probability),
    ("Baseline test", baseline_test_probability),
    ("Volume Infill test", infill_test_probability)
]

for name, probability_volume in probability_diagnostics:
    print_probability_diagnostics(name, probability_volume)


# ============================================================
# 6. Extract provisional particle centers
# ============================================================

# WriteYourCodeHere — use 0.5 only as a temporary diagnostic threshold.
# Task 6 will select model-specific thresholds using validation data.
baseline_predicted_centers, baseline_test_binary, baseline_component_sizes = extract_particle_centers(
    baseline_test_probability,
    threshold=PROVISIONAL_THRESHOLD,
    min_component_voxels=MIN_COMPONENT_VOXELS
)

infill_predicted_centers, infill_test_binary, infill_component_sizes = extract_particle_centers(
    infill_test_probability,
    threshold=PROVISIONAL_THRESHOLD,
    min_component_voxels=MIN_COMPONENT_VOXELS
)


# WriteYourCodeHere — explain whether detections disappeared because of the
# probability threshold or the minimum-component-size filter.
def print_provisional_diagnostics(model_name, probability_volume, predicted_centers):
    voxels_above_threshold = int((probability_volume >= PROVISIONAL_THRESHOLD).sum())

    print(f"\n{model_name}:")
    print(f"  Maximum probability: {probability_volume.max():.4f}")
    print(f"  Voxels with P >= {PROVISIONAL_THRESHOLD:.2f}: {voxels_above_threshold:,}")
    print(f"  Components retained after the {MIN_COMPONENT_VOXELS}-voxel filter: {len(predicted_centers)}")


print_provisional_diagnostics(
    "Baseline",
    baseline_test_probability,
    baseline_predicted_centers
)

print_provisional_diagnostics(
    "Volume Infill",
    infill_test_probability,
    infill_predicted_centers
)


# ============================================================
# 7. Visualize whole-volume probability maps
# ============================================================

# WriteYourCodeHere — choose a display slice from model predictions only.
# Test ground-truth centers are not used to select this slice.
combined_probability = np.maximum(baseline_test_probability, infill_test_probability)
display_z = int(np.argmax(combined_probability.max(axis=(1, 2))))

test_view = test_tomogram[display_z]
baseline_probability_view = baseline_test_probability[display_z]
infill_probability_view = infill_test_probability[display_z]

input_min, input_max = np.percentile(test_view, [2, 98])

# Use one shared probability range so both model maps remain comparable.
probability_min = min(
    float(np.percentile(baseline_test_probability, 1)),
    float(np.percentile(infill_test_probability, 1))
)

probability_max = max(
    float(np.percentile(baseline_test_probability, 99.9)),
    float(np.percentile(infill_test_probability, 99.9))
)

if probability_max <= probability_min:
    probability_max = probability_min + 1e-3

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.6), dpi=140, constrained_layout=True)

axes[0].imshow(
    test_view,
    cmap="magma",
    origin="lower",
    interpolation="bilinear",
    vmin=input_min,
    vmax=input_max
)

axes[0].set_title(
    f"Test tomogram input\nz={display_z}"
)

baseline_probability_image = axes[1].imshow(
    baseline_probability_view,
    cmap="viridis",
    origin="lower",
    interpolation="bilinear",
    vmin=probability_min,
    vmax=probability_max
)

axes[1].set_title(
    "Baseline probability\n"
    f"max={baseline_test_probability.max():.4f}"
)

axes[2].imshow(
    infill_probability_view,
    cmap="viridis",
    origin="lower",
    interpolation="bilinear",
    vmin=probability_min,
    vmax=probability_max
)

axes[2].set_title(
    "Volume Infill probability\n"
    f"max={infill_test_probability.max():.4f}"
)

for axis in axes:
    axis.set_xlabel("x")
    axis.set_ylabel("y")
    axis.set_aspect("equal")

fig.colorbar(
    baseline_probability_image,
    ax=[axes[1], axes[2]],
    fraction=0.03,
    pad=0.02,
    label="Predicted particle probability"
)

fig.suptitle(
    "Whole-volume probability maps before threshold selection",
    fontsize=14
)

plt.show()


# ============================================================
# 8. Plot provisional 3D centers only when detections exist
# ============================================================

# WriteYourCodeHere — do not create empty 3D coordinate plots.
total_provisional_detections = (
    len(baseline_predicted_centers)
    + len(infill_predicted_centers)
)

if total_provisional_detections == 0:
    print(
        "\n3D visualization skipped: neither model produced a component "
        "that passed the provisional threshold and size filter. "
        "Task 6 will select model-specific thresholds using validation data."
    )

else:
    fig = plt.figure(figsize=(10, 4.5), dpi=140)

    plot_settings = [
        (
            baseline_predicted_centers,
            "Baseline provisional centers",
            "#64748b"
        ),
        (
            infill_predicted_centers,
            "Volume Infill provisional centers",
            "#06b6d4"
        )
    ]

    for plot_index, (centers, title, color) in enumerate(plot_settings, start=1):
        axis = fig.add_subplot(1, 2, plot_index, projection="3d")

        if len(centers):
            axis.scatter(
                centers[:, 2],
                centers[:, 1],
                centers[:, 0],
                s=45,
                color=color,
                edgecolors="black",
                linewidths=0.4
            )
        else:
            axis.text2D(
                0.5,
                0.5,
                "No provisional detection",
                transform=axis.transAxes,
                ha="center",
                va="center"
            )

        axis.set_xlim(0, test_tomogram.shape[2])
        axis.set_ylim(0, test_tomogram.shape[1])
        axis.set_zlim(0, test_tomogram.shape[0])
        axis.set_xlabel("x")
        axis.set_ylabel("y")
        axis.set_zlabel("z")
        axis.set_title(f"{title}\nn={len(centers)}")
        axis.set_box_aspect(test_tomogram.shape[::-1])

    fig.suptitle(
        "Provisional particle coordinates; ground truth is reserved for Task 6",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# Checkpoint
# ============================================================

probability_pairs = [
    (
        baseline_val_probability,
        validation_tomogram
    ),
    (
        infill_val_probability,
        validation_tomogram
    ),
    (
        baseline_test_probability,
        test_tomogram
    ),
    (
        infill_test_probability,
        test_tomogram
    )
]

# Check 1: every probability volume must match its input tomogram.
assert all(
    probability.shape == volume.shape
    for probability, volume
    in probability_pairs
)

# Check 2: all probabilities must be finite and remain in [0,1].
assert all(
    np.isfinite(probability).all()
    and probability.min() >= 0
    and probability.max() <= 1
    for probability, _
    in probability_pairs
)

# Check 3: every voxel must receive at least one sliding-window prediction.
assert all(
    np.all(count > 0)
    for count in [
        baseline_val_overlap_count,
        infill_val_overlap_count,
        baseline_test_overlap_count,
        infill_test_overlap_count
    ]
)

# Check 4: models must use identical sliding-window coverage.
assert np.array_equal(
    baseline_val_overlap_count,
    infill_val_overlap_count
)

assert np.array_equal(
    baseline_test_overlap_count,
    infill_test_overlap_count
)

# Check 5: retained component centers and sizes must remain valid.
for centers, sizes, binary in [
    (
        baseline_predicted_centers,
        baseline_component_sizes,
        baseline_test_binary
    ),
    (
        infill_predicted_centers,
        infill_component_sizes,
        infill_test_binary
    )
]:
    assert len(centers) == len(sizes)

    assert all(
        size >= MIN_COMPONENT_VOXELS
        for size in sizes
    )

    assert all(
        np.all(center >= 0)
        and np.all(
            center
            < np.asarray(
                test_tomogram.shape
            )
        )
        for center in centers
    )

    assert binary.shape == test_tomogram.shape
    assert binary.dtype == bool

# Check 6: test data must remain independent of training and validation.
assert test_provenance == "independent_test_tomogram"
assert TEST_SEED != SEED
assert TEST_SEED != VALIDATION_SEED

print(
    "Checkpoint passed: every voxel was covered, probability volumes are valid, "
    "and all provisional centers remain inside the independent test volume."
)

## Part 6 — Detection Evaluation and Volume Infill Comparison

A probability map or visually convincing segmentation does not directly measure particle-detection performance. We must compare predicted coordinates with ground-truth particle centers using a one-to-one matching rule.

Part 6 will select detection thresholds on the validation tomogram and then report the final baseline-versus-Volume-Infill comparison on the unseen test tomogram.

### 6.1 Distance-Based One-to-One Matching

Let the predicted centers be $\hat{\mathcal{C}}={\hat{\mathbf{c}}_1,\ldots,\hat{\mathbf{c}}_P}$ and the ground-truth centers be $\mathcal{C}={\mathbf{c}_1,\ldots,\mathbf{c}_G}$.

A prediction can match a ground-truth particle only when:

$$|\hat{\mathbf{c}}_i-\mathbf{c}_j|*2\leq r*{\mathrm{match}}$$

Each prediction and each ground-truth center may be used at most once. One-to-one matching prevents several nearby predictions from all being counted as correct detections of the same particle.

The default matching radius is:

$$r_{\mathrm{match}}=5\text{ voxels}$$

This is slightly larger than the approximate synthetic particle radius and allows small localization errors without accepting detections from unrelated regions.

After matching:

* **True positive (TP):** a prediction matched to one ground-truth particle;
* **False positive (FP):** a prediction with no accepted match;
* **False negative (FN):** a ground-truth particle with no accepted prediction.

### 6.2 Detection Metrics

Precision measures the fraction of predicted particles that are correct:

$$\mathrm{Precision}=\frac{TP}{TP+FP}$$

Recall measures the fraction of ground-truth particles that are detected:

$$\mathrm{Recall}=\frac{TP}{TP+FN}$$

F1-score balances precision and recall:

$$F1=\frac{2,\mathrm{Precision},\mathrm{Recall}}{\mathrm{Precision}+\mathrm{Recall}}$$

For all matched pairs, mean localization error is:

$$E_{\mathrm{loc}}=\frac{1}{TP}\sum_{(\hat{\mathbf{c}},\mathbf{c})\in\mathcal{M}}|\hat{\mathbf{c}}-\mathbf{c}|_2$$

A detector can achieve high recall by producing many candidates, but this may reduce precision. F1-score is therefore the primary comparison metric, while localization error describes coordinate accuracy among correct detections.

### 6.3 Selecting Thresholds Without Test Leakage

The threshold $\tau$ strongly affects the number of connected components. It must not be selected using test results.

For each model, Part 6 will:

1. sweep a shared threshold grid on the validation tomogram;
2. calculate validation precision, recall, and F1-score;
3. select the threshold with the highest validation F1-score;
4. apply that fixed threshold to the test probability volume;
5. report the final test metrics.

The two models may select different thresholds because their probability calibration may differ. This remains a fair comparison because both use the same validation volume, threshold grid, matching rule, and selection criterion.

### 6.4 Interpreting the Comparison

The central experiment compares:

| Model             | Only difference                              |
| ----------------- | -------------------------------------------- |
| **Baseline**      | Trained without infilled samples             |
| **Volume Infill** | Trained with the additional infilled samples |

If Volume Infill improves recall, the extra particle variants may have helped the model recognize more appearances. If precision decreases, some inserted patterns may have encouraged additional false positives. An improvement in training loss alone is not sufficient; the method should improve validation-selected test detection.

Because this tutorial uses one small synthetic training, validation, and test split, the result demonstrates the evaluation procedure rather than establishing a statistically general conclusion. Repeated seeds would be required for a stronger scientific claim.

---

## Task 6 — Evaluate the Effect of Volume Infill

### Scenario

You now have validation and test probability volumes from both models. Your goal is to select thresholds without test leakage, perform one-to-one coordinate matching, and determine whether Volume Infill improves particle detection.

### What You Will Do

In the following code cell, you will:

1. define distance-constrained one-to-one matching;
2. calculate TP, FP, FN, precision, recall, F1-score, and localization error;
3. sweep probability thresholds on the validation tomogram;
4. select the best validation threshold for each model;
5. extract final test coordinates using the selected thresholds;
6. calculate baseline and Volume Infill test metrics;
7. classify detections as TP, FP, or FN;
8. compare the two models visually and numerically.

### Main Editable Parameters

| Parameter              |                       Default value | Meaning                                     |
| ---------------------- | ----------------------------------: | ------------------------------------------- |
| `THRESHOLDS`           | `0.20` to `0.80` in steps of `0.05` | Validation threshold candidates             |
| `MATCH_RADIUS`         |                                 `5` | Maximum distance for a correct detection    |
| `MIN_COMPONENT_VOXELS` |                                 `5` | Component-size filter inherited from Part 5 |

### Expected Outputs

The final results table will contain:

| Model         |  Selected threshold | Predicted particles |         TP |         FP |         FN |  Precision |     Recall |         F1 | Localization error |
| ------------- | ------------------: | ------------------: | ---------: | ---------: | ---------: | ---------: | ---------: | ---------: | -----------------: |
| Baseline      | Validation-selected |          Calculated | Calculated | Calculated | Calculated | Calculated | Calculated | Calculated |         Calculated |
| Volume Infill | Validation-selected |          Calculated | Calculated | Calculated | Calculated | Calculated | Calculated | Calculated |         Calculated |

The expected visualizations are:

* validation F1-score against probability threshold;
* final baseline and Volume Infill metric comparison;
* 3D test detections marked as TP, FP, and FN.

### Checkpoint

The code should confirm that:

* thresholds are selected using validation data only;
* every prediction and ground-truth center participates in at most one match;
* $TP+FN$ equals the number of ground-truth particles;
* $TP+FP$ equals the number of predicted particles;
* precision, recall, and F1-score remain in $[0,1]$;
* localization error is calculated only from matched pairs;
* both models use the same test tomogram, matching radius, and component filter.

The resulting comparison will provide the main experimental conclusion of this notebook.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from scipy.optimize import linear_sum_assignment
    SCIPY_MATCHING = True
except ModuleNotFoundError:
    SCIPY_MATCHING = False

# ============================================================
# Task 6 — Evaluate the effect of Volume Infill
# ============================================================

# Task 6 selects thresholds using validation data, freezes those thresholds,
# and then evaluates both models on the unseen test tomogram.

# WriteYourCodeHere — adjust matching tolerance and validation-only threshold resolution.
MATCH_RADIUS = 5.0
THRESHOLD_QUANTILE_MIN = 0.80
THRESHOLD_QUANTILE_MAX = 0.9999
NUM_THRESHOLD_CANDIDATES = 41

assert 0 <= THRESHOLD_QUANTILE_MIN < THRESHOLD_QUANTILE_MAX <= 1


# ============================================================
# 1. Construct validation-only threshold candidates
# ============================================================

# WriteYourCodeHere — derive threshold values from one model's validation
# probability distribution without reading its test predictions.
def build_validation_thresholds(probability_volume):
    quantiles = np.linspace(
        THRESHOLD_QUANTILE_MIN,
        THRESHOLD_QUANTILE_MAX,
        NUM_THRESHOLD_CANDIDATES
    )

    thresholds = np.unique(
        np.quantile(
            probability_volume,
            quantiles
        ).astype(np.float32)
    )

    if len(thresholds) < 2:
        center = float(
            probability_volume.mean()
        )

        thresholds = np.array([
            max(0.0, center - 1e-3),
            min(1.0, center + 1e-3)
        ], dtype=np.float32)

    return thresholds


# ============================================================
# 2. Define one-to-one particle matching
# ============================================================

# WriteYourCodeHere — match predictions and ground truth under a maximum
# Euclidean distance while preventing duplicate matches.
def match_particle_centers(predicted_centers, ground_truth_centers, match_radius=MATCH_RADIUS):
    predicted = np.asarray(
        predicted_centers,
        dtype=np.float32
    ).reshape(-1, 3)

    ground_truth = np.asarray(
        ground_truth_centers,
        dtype=np.float32
    ).reshape(-1, 3)

    prediction_count = len(predicted)
    ground_truth_count = len(ground_truth)

    if prediction_count == 0 or ground_truth_count == 0:
        matched_pred = np.empty(
            0,
            dtype=int
        )

        matched_gt = np.empty(
            0,
            dtype=int
        )

        distances = np.empty(
            0,
            dtype=np.float32
        )

    else:
        distance_matrix = np.linalg.norm(
            predicted[:, None, :]
            - ground_truth[None, :, :],
            axis=2
        )

        if SCIPY_MATCHING:
            # Dummy columns allow predictions to remain unmatched.
            # The large dummy cost prioritizes the maximum number of
            # valid matches before minimizing localization distance.
            unmatched_cost = (
                max(
                    prediction_count,
                    ground_truth_count
                )
                + 1
            ) * (
                match_radius + 1
            )

            invalid_cost = (
                2 * unmatched_cost
            )

            cost_matrix = np.full(
                (
                    prediction_count,
                    ground_truth_count
                    + prediction_count
                ),
                unmatched_cost,
                dtype=np.float64
            )

            cost_matrix[
                :,
                :ground_truth_count
            ] = np.where(
                distance_matrix
                <= match_radius,
                distance_matrix,
                invalid_cost
            )

            row_indices, column_indices = linear_sum_assignment(
                cost_matrix
            )

            checked_columns = np.minimum(
                column_indices,
                ground_truth_count - 1
            )

            valid_matches = (
                column_indices
                < ground_truth_count
            ) & (
                distance_matrix[
                    row_indices,
                    checked_columns
                ]
                <= match_radius
            )

            matched_pred = row_indices[
                valid_matches
            ].astype(int)

            matched_gt = column_indices[
                valid_matches
            ].astype(int)

        else:
            # Portable maximum-cardinality bipartite matching fallback.
            adjacency = [
                list(
                    np.where(
                        distance_matrix[
                            prediction_index
                        ]
                        <= match_radius
                    )[0]
                )
                for prediction_index
                in range(prediction_count)
            ]

            adjacency = [
                sorted(
                    neighbors,
                    key=lambda gt_index: distance_matrix[
                        prediction_index,
                        gt_index
                    ]
                )
                for prediction_index, neighbors
                in enumerate(adjacency)
            ]

            ground_truth_to_prediction = {}

            def find_augmenting_path(prediction_index, visited_ground_truth):
                for ground_truth_index in adjacency[
                    prediction_index
                ]:
                    if ground_truth_index in visited_ground_truth:
                        continue

                    visited_ground_truth.add(
                        ground_truth_index
                    )

                    if (
                        ground_truth_index
                        not in ground_truth_to_prediction
                    ) or find_augmenting_path(
                        ground_truth_to_prediction[
                            ground_truth_index
                        ],
                        visited_ground_truth
                    ):
                        ground_truth_to_prediction[
                            ground_truth_index
                        ] = prediction_index

                        return True

                return False

            prediction_order = sorted(
                range(prediction_count),
                key=lambda index: (
                    min(
                        distance_matrix[
                            index
                        ]
                    )
                    if ground_truth_count
                    else np.inf
                )
            )

            for prediction_index in prediction_order:
                find_augmenting_path(
                    prediction_index,
                    set()
                )

            matched_gt = np.asarray(
                sorted(
                    ground_truth_to_prediction
                ),
                dtype=int
            )

            matched_pred = np.asarray([
                ground_truth_to_prediction[
                    index
                ]
                for index in matched_gt
            ], dtype=int)

        distances = distance_matrix[
            matched_pred,
            matched_gt
        ].astype(np.float32)

    tp = len(matched_pred)
    fp = prediction_count - tp
    fn = ground_truth_count - tp

    precision = (
        tp / prediction_count
        if prediction_count
        else 0.0
    )

    recall = (
        tp / ground_truth_count
        if ground_truth_count
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    matched_pred_set = set(
        matched_pred.tolist()
    )

    matched_gt_set = set(
        matched_gt.tolist()
    )

    return {
        "predicted_count": prediction_count,
        "ground_truth_count": ground_truth_count,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "localization_error": (
            float(
                distances.mean()
            )
            if tp
            else np.nan
        ),
        "matched_pred_indices": matched_pred,
        "matched_gt_indices": matched_gt,
        "matched_distances": distances,
        "unmatched_pred_indices": np.asarray([
            index
            for index
            in range(prediction_count)
            if index not in matched_pred_set
        ], dtype=int),
        "unmatched_gt_indices": np.asarray([
            index
            for index
            in range(ground_truth_count)
            if index not in matched_gt_set
        ], dtype=int)
    }


# ============================================================
# 3. Evaluate validation threshold candidates
# ============================================================

# WriteYourCodeHere — calculate validation detections and metrics for every
# candidate threshold.
def evaluate_threshold_grid(probability_volume, ground_truth_centers, thresholds):
    records = []

    for threshold in thresholds:
        centers, binary_mask, component_sizes = extract_particle_centers(
            probability_volume,
            threshold=float(threshold),
            min_component_voxels=MIN_COMPONENT_VOXELS
        )

        metrics = match_particle_centers(
            centers,
            ground_truth_centers,
            MATCH_RADIUS
        )

        records.append({
            "threshold": float(threshold),
            "centers": centers,
            "binary_mask": binary_mask,
            "component_sizes": component_sizes,
            **metrics
        })

    return records


# WriteYourCodeHere — maximize validation F1. Ties prefer precision, recall,
# and then the higher threshold.
def select_best_validation_record(records):
    return max(
        records,
        key=lambda record: (
            record["f1"],
            record["precision"],
            record["recall"],
            record["threshold"]
        )
    )


# WriteYourCodeHere — create model-specific threshold candidates using only
# the corresponding validation probability volume.
baseline_thresholds = build_validation_thresholds(
    baseline_val_probability
)

infill_thresholds = build_validation_thresholds(
    infill_val_probability
)

baseline_validation_records = evaluate_threshold_grid(
    baseline_val_probability,
    validation_centers,
    baseline_thresholds
)

infill_validation_records = evaluate_threshold_grid(
    infill_val_probability,
    validation_centers,
    infill_thresholds
)

baseline_best_validation = select_best_validation_record(
    baseline_validation_records
)

infill_best_validation = select_best_validation_record(
    infill_validation_records
)

baseline_selected_threshold = baseline_best_validation[
    "threshold"
]

infill_selected_threshold = infill_best_validation[
    "threshold"
]

print(
    f"Baseline validation-selected threshold: "
    f"{baseline_selected_threshold:.6f}, "
    f"validation F1="
    f"{baseline_best_validation['f1']:.3f}"
)

print(
    f"Volume Infill validation-selected threshold: "
    f"{infill_selected_threshold:.6f}, "
    f"validation F1="
    f"{infill_best_validation['f1']:.3f}"
)


# ============================================================
# 4. Freeze thresholds and evaluate the unseen test volume
# ============================================================

# The threshold dictionary is created before test ground truth is accessed.
selected_thresholds = {
    "Baseline": baseline_selected_threshold,
    "Volume Infill": infill_selected_threshold
}

# WriteYourCodeHere — apply the fixed validation-selected thresholds to test
# probability volumes.
baseline_final_centers, baseline_final_binary, baseline_final_component_sizes = extract_particle_centers(
    baseline_test_probability,
    baseline_selected_threshold,
    MIN_COMPONENT_VOXELS
)

infill_final_centers, infill_final_binary, infill_final_component_sizes = extract_particle_centers(
    infill_test_probability,
    infill_selected_threshold,
    MIN_COMPONENT_VOXELS
)

# Test ground truth is used only after both thresholds have been frozen.
baseline_test_metrics = match_particle_centers(
    baseline_final_centers,
    test_centers,
    MATCH_RADIUS
)

infill_test_metrics = match_particle_centers(
    infill_final_centers,
    test_centers,
    MATCH_RADIUS
)


# ============================================================
# 5. Print final test results
# ============================================================

# WriteYourCodeHere — report both models using the same metric definitions.
result_rows = [
    (
        "Baseline",
        baseline_selected_threshold,
        baseline_test_metrics
    ),
    (
        "Volume Infill",
        infill_selected_threshold,
        infill_test_metrics
    )
]

header = (
    f"{'Model':<16}"
    f"{'Threshold':>11}"
    f"{'Pred':>7}"
    f"{'TP':>5}"
    f"{'FP':>5}"
    f"{'FN':>5}"
    f"{'Precision':>11}"
    f"{'Recall':>9}"
    f"{'F1':>8}"
    f"{'Loc. error':>12}"
)

print("\nFinal test results")
print(header)
print("-" * len(header))

for model_name, threshold, metrics in result_rows:
    localization_text = (
        f"{metrics['localization_error']:.3f}"
        if np.isfinite(
            metrics[
                "localization_error"
            ]
        )
        else "N/A"
    )

    print(
        f"{model_name:<16}"
        f"{threshold:>11.6f}"
        f"{metrics['predicted_count']:>7}"
        f"{metrics['tp']:>5}"
        f"{metrics['fp']:>5}"
        f"{metrics['fn']:>5}"
        f"{metrics['precision']:>11.3f}"
        f"{metrics['recall']:>9.3f}"
        f"{metrics['f1']:>8.3f}"
        f"{localization_text:>12}"
    )


# ============================================================
# 6. Visualize validation threshold selection
# ============================================================

# WriteYourCodeHere — display validation F1 against each model's threshold.
fig, ax = plt.subplots(
    figsize=(6.5, 4),
    dpi=140
)

ax.plot(
    [
        record["threshold"]
        for record
        in baseline_validation_records
    ],
    [
        record["f1"]
        for record
        in baseline_validation_records
    ],
    marker="o",
    markersize=3,
    label="Baseline",
    color="#64748b"
)

ax.plot(
    [
        record["threshold"]
        for record
        in infill_validation_records
    ],
    [
        record["f1"]
        for record
        in infill_validation_records
    ],
    marker="o",
    markersize=3,
    label="Volume Infill",
    color="#06b6d4"
)

ax.axvline(
    baseline_selected_threshold,
    color="#64748b",
    linestyle="--",
    alpha=0.7
)

ax.axvline(
    infill_selected_threshold,
    color="#06b6d4",
    linestyle="--",
    alpha=0.7
)

ax.set_xlabel(
    "Validation probability threshold"
)

ax.set_ylabel(
    "Validation F1-score"
)

ax.set_title(
    "Validation-only threshold selection"
)

ax.legend()
ax.grid(alpha=0.2)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()


# ============================================================
# 7. Compare final test metrics
# ============================================================

# WriteYourCodeHere — compare precision, recall, and F1 on the same test volume.
metric_names = [
    "precision",
    "recall",
    "f1"
]

x_positions = np.arange(
    len(metric_names)
)

bar_width = 0.34

baseline_metric_values = [
    baseline_test_metrics[
        name
    ]
    for name in metric_names
]

infill_metric_values = [
    infill_test_metrics[
        name
    ]
    for name in metric_names
]

fig, ax = plt.subplots(
    figsize=(6.5, 4),
    dpi=140
)

baseline_bars = ax.bar(
    x_positions - bar_width / 2,
    baseline_metric_values,
    bar_width,
    label="Baseline",
    color="#64748b"
)

infill_bars = ax.bar(
    x_positions + bar_width / 2,
    infill_metric_values,
    bar_width,
    label="Volume Infill",
    color="#06b6d4"
)

ax.bar_label(
    baseline_bars,
    labels=[
        f"{value:.2f}"
        for value
        in baseline_metric_values
    ],
    padding=3
)

ax.bar_label(
    infill_bars,
    labels=[
        f"{value:.2f}"
        for value
        in infill_metric_values
    ],
    padding=3
)

ax.set_xticks(
    x_positions,
    [
        "Precision",
        "Recall",
        "F1"
    ]
)

ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")

ax.set_title(
    "Final detection performance "
    "on the unseen test tomogram"
)

ax.legend()
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()


# ============================================================
# 8. Visualize TP, FP, and FN coordinates
# ============================================================

# WriteYourCodeHere — classify final predictions and missed ground-truth
# particles for visual interpretation.
def plot_detection_outcomes(axis, predicted_centers, ground_truth_centers, metrics, title):
    matched_predictions = (
        predicted_centers[
            metrics[
                "matched_pred_indices"
            ]
        ]
        if metrics["tp"]
        else np.empty((0, 3))
    )

    false_positives = (
        predicted_centers[
            metrics[
                "unmatched_pred_indices"
            ]
        ]
        if metrics["fp"]
        else np.empty((0, 3))
    )

    false_negatives = (
        ground_truth_centers[
            metrics[
                "unmatched_gt_indices"
            ]
        ]
        if metrics["fn"]
        else np.empty((0, 3))
    )

    if len(matched_predictions):
        axis.scatter(
            matched_predictions[:, 2],
            matched_predictions[:, 1],
            matched_predictions[:, 0],
            s=50,
            color="#22c55e",
            marker="o",
            label="TP prediction"
        )

    if len(false_positives):
        axis.scatter(
            false_positives[:, 2],
            false_positives[:, 1],
            false_positives[:, 0],
            s=55,
            color="#f97316",
            marker="^",
            label="FP prediction"
        )

    if len(false_negatives):
        axis.scatter(
            false_negatives[:, 2],
            false_negatives[:, 1],
            false_negatives[:, 0],
            s=60,
            color="#ef4444",
            marker="x",
            linewidths=2,
            label="FN ground truth"
        )

    axis.set_xlim(
        0,
        test_tomogram.shape[2]
    )

    axis.set_ylim(
        0,
        test_tomogram.shape[1]
    )

    axis.set_zlim(
        0,
        test_tomogram.shape[0]
    )

    axis.set_xlabel("x")
    axis.set_ylabel("y")
    axis.set_zlabel("z")
    axis.set_title(title)

    axis.set_box_aspect(
        test_tomogram.shape[::-1]
    )

    handles, labels = axis.get_legend_handles_labels()

    if handles:
        axis.legend(
            loc="upper left",
            fontsize=8
        )


fig = plt.figure(
    figsize=(10, 4.8),
    dpi=140
)

baseline_axis = fig.add_subplot(
    1,
    2,
    1,
    projection="3d"
)

infill_axis = fig.add_subplot(
    1,
    2,
    2,
    projection="3d"
)

plot_detection_outcomes(
    baseline_axis,
    baseline_final_centers,
    test_centers,
    baseline_test_metrics,
    (
        f"Baseline: "
        f"TP={baseline_test_metrics['tp']}, "
        f"FP={baseline_test_metrics['fp']}, "
        f"FN={baseline_test_metrics['fn']}"
    )
)

plot_detection_outcomes(
    infill_axis,
    infill_final_centers,
    test_centers,
    infill_test_metrics,
    (
        f"Volume Infill: "
        f"TP={infill_test_metrics['tp']}, "
        f"FP={infill_test_metrics['fp']}, "
        f"FN={infill_test_metrics['fn']}"
    )
)

fig.suptitle(
    "Final test detections after "
    "validation-only threshold selection",
    fontsize=14
)

plt.tight_layout()
plt.show()


# ============================================================
# Checkpoint
# ============================================================

# Check 1: TP, FP, and FN must agree with the numbers of predictions and targets.
for centers, metrics in [
    (
        baseline_final_centers,
        baseline_test_metrics
    ),
    (
        infill_final_centers,
        infill_test_metrics
    )
]:
    assert (
        metrics["tp"]
        + metrics["fn"]
        == len(test_centers)
    )

    assert (
        metrics["tp"]
        + metrics["fp"]
        == len(centers)
    )

    # Check 2: no prediction or ground-truth particle may be matched twice.
    assert (
        len(
            np.unique(
                metrics[
                    "matched_pred_indices"
                ]
            )
        )
        == metrics["tp"]
    )

    assert (
        len(
            np.unique(
                metrics[
                    "matched_gt_indices"
                ]
            )
        )
        == metrics["tp"]
    )

    # Check 3: all metrics and accepted match distances must be valid.
    assert all(
        0 <= metrics[name] <= 1
        for name in [
            "precision",
            "recall",
            "f1"
        ]
    )

    assert np.all(
        metrics[
            "matched_distances"
        ]
        <= MATCH_RADIUS + 1e-6
    )

    if metrics["tp"]:
        assert np.isfinite(
            metrics[
                "localization_error"
            ]
        )
    else:
        assert np.isnan(
            metrics[
                "localization_error"
            ]
        )

# Check 4: selected thresholds must come from validation-only candidate arrays.
assert baseline_selected_threshold in baseline_thresholds
assert infill_selected_threshold in infill_thresholds

assert selected_thresholds == {
    "Baseline": baseline_selected_threshold,
    "Volume Infill": infill_selected_threshold
}

# Check 5: both models use the same test volume, matching rule, and component filter.
assert TEST_SEED != VALIDATION_SEED
assert MATCH_RADIUS == 5.0
assert MIN_COMPONENT_VOXELS == 5

print(
    "Checkpoint passed: thresholds came only from validation data, "
    "matching is one-to-one, and final test metrics are internally consistent."
)

## Part 7 — Summary, Limitations, and Next Steps

This notebook demonstrated a complete educational pipeline for few-shot particle detection in synthetic CryoET tomograms. Starting from only a small set of annotated particle centers, it constructed spherical weak labels, extracted aligned positive and verified-background subvolumes, generated additional training samples through SaSi-inspired Volume Infill, trained two lightweight 3D detectors, reconstructed whole-volume probability maps, and evaluated the resulting particle coordinates.

The central comparison kept the detector architecture, initialization, optimizer, number of training steps, validation volume, test volume, component filter, and matching rule fixed. The intended experimental difference was the training data: the baseline model used only the original few-shot samples, whereas the Volume Infill model also used transformed particles inserted into new verified backgrounds.

### 7.1 Interpreting the Final Comparison

The final conclusion should be based primarily on the validation-selected test F1-score rather than training loss or visual appearance alone.

| Result pattern | Possible interpretation |
|---|---|
| Higher recall after Volume Infill | Additional particle variants helped the model detect more true particles |
| Higher precision after Volume Infill | The augmented samples improved discrimination between particles and background |
| Higher F1-score after Volume Infill | The precision-recall balance improved under the shared evaluation protocol |
| Lower localization error | Correct detections were placed closer to the true particle centers |
| Better training loss but similar test F1 | The model fitted the augmented samples without a clear detection benefit |
| Higher recall but lower precision | Volume Infill found more particles but also introduced more false positives |

The two models may use different validation-selected thresholds because their probability calibration can differ. This does not make the comparison unfair: each threshold is selected from the corresponding validation probability volume using the same quantile range, F1 criterion, component filter, and matching radius. The unseen test results are not used during threshold selection.

### 7.2 What Volume Infill Adds

Ordinary rotation, flipping, or shifting modifies an existing training crop but does not create a new particle-containing context. Volume Infill performs an additional operation by inserting a transformed annotated particle into a verified background region. It therefore increases both particle-appearance variation and the proportion of training samples containing particles.

This process does not create new manual annotations or a new biological particle class. Every infilled target is derived from one of the original few-shot annotations. Its purpose is to reuse limited supervision more effectively.

### 7.3 Limitations

The experiment is intentionally simplified and should not be interpreted as a full reproduction of SaSi or as evidence of performance on real CryoET datasets.

First, the tomograms, particles, noise, and complete ground-truth coordinates are synthetic. Complete particle centers are used to verify background crops and evaluate detections, but such privileged information is generally unavailable in real few-shot CryoET workflows.

Second, the tutorial implements a simplified Volume Infill procedure based on exact 90-degree rotations, axis flips, integer shifts, and soft spatial blending. It does not reproduce the complete AugMix-inspired transformation and stochastic mixing strategy used by SaSi.

Third, spherical weak labels approximate a small region around each annotated center and do not represent exact particle boundaries. The voxel probabilities should therefore be interpreted primarily as a route to coordinate localization rather than as precise biological segmentation.

Fourth, both models are trained on one small synthetic split using a Tiny 3D U-Net. A single training, validation, and test seed cannot establish a statistically general improvement. The probability outputs may also be poorly calibrated, which is why threshold selection must use independent validation data.

Finally, connected-component localization may merge nearby probability regions or split one weak particle response into several components. Detection results can therefore depend on the probability threshold, minimum component size, particle spacing, and matching radius.

### 7.4 Possible Extensions

A stronger follow-up experiment could:

1. repeat the complete comparison across multiple random seeds and report mean and standard deviation;
2. test several values of `N_SHOT`, subvolume size, blending radius, and number of infilled samples;
3. compare Volume Infill with ordinary rotation-and-flip augmentation;
4. evaluate alternative post-processing methods such as local-maximum detection or non-maximum suppression;
5. calibrate output probabilities using independent validation data;
6. introduce more realistic CryoET degradations, particle crowding, and missing-wedge effects;
7. replace the synthetic tomograms with real annotated CryoET volumes;
8. compare the Tiny 3D U-Net with a stronger 3D detection architecture.

### Final Takeaway

The main lesson is not that synthetic Volume Infill must always outperform the baseline. The notebook demonstrates how a few-shot augmentation method should be tested: training data must remain separated from validation and test volumes, image-label transformations must remain aligned, optimizer updates must be controlled, thresholds must be selected without test leakage, and final performance must be measured using one-to-one particle-coordinate matching.

Within this controlled pipeline, the Task 6 results indicate whether the additional Volume Infill samples improved detection for the current synthetic split. Repeated experiments and real CryoET data would be required before making a broader scientific claim.